# Notebook 03: Monthly & Annual LST Temporal Analysis

## Overview
This notebook extracts and analyzes Land Surface Temperature (LST) data from MODIS Terra (MOD11A1) and Aqua (MYD11A1) satellites using Google Earth Engine. It processes all cities with delineated AOIs from **Notebook 01** and performs comprehensive temporal trend analysis.

## Prerequisites
- ✅ Completed **Notebook 01** (AOI Delineation)
- ✅ Configured `config.py` with valid GEE credentials
- ✅ Activated virtual environment with required packages

## Key Capabilities
- **Multi-City Processing**: Automatically detects and processes all AOI outputs
- **Temporal Scope**: 2006-2025 (configurable)
- **LST Acquisition**: Day, Night, or Both (set in `config.py`)
- **Quality Assurance**: Hierarchical QA filtering (Strict → Moderate)
- **Parallel Processing**: Dask-based parallelization for efficiency
- **Statistical Analysis**: Mann-Kendall trend tests, linear regression, climatology
- **Stakeholder Validation**: Compare perceptions with measured trends

## Workflow Structure
1. **Setup & Authentication** → Initialize GEE and Dask
2. **Load Configuration** → Import settings from `config.py`
3. **Load AOI Data** → Auto-load city boundaries
4. **LST Extraction** → Download MODIS LST with QA filtering
5. **Visualize LST Maps** → Generate monthly/annual maps
6. **Compute Statistics** → Calculate temporal metrics
7. **Trend Analysis** → Perform statistical tests
8. **Cross-City Comparison** → Create heatmaps
9. **Stakeholder Validation** → Compare perceptions vs. reality

## Output Directories
```
data/outputs/03_trend_analysis_monthly_annual/
├── City_*/
│   └── LST_Temporal_Analysis/
│       ├── Day/ (or Night/)
│       │   ├── GeoTIFF/
│       │   ├── Figures/
│       │   └── Statistics/
│       └── Cross_City_Statistics/
```

## Next Steps
After completing this notebook, proceed to **Notebook 04** for daily LST analysis and diurnal temperature variations.

## Mathematical Background: Trend Analysis

### 1. Linear Regression Model
For each city and time period (day/night), we fit an ordinary least squares (OLS) regression:

$$\text{LST}_t = \beta_0 + \beta_1 \cdot t + \epsilon_t$$

where:
- $\text{LST}_t$ = median LST at time $t$ (°C)
- $t$ = year (2006, 2007, ..., 2025)
- $\beta_0$ = intercept (baseline temperature)
- $\beta_1$ = slope (°C/year)
- $\epsilon_t$ = residual error

### 2. Warming/Cooling Rate Metrics

**Per Decade Rate:**
$$\Delta T_{decade} = 10 \cdot \beta_1$$

**Total Study Period Change:**
$$\Delta T_{total} = \beta_1 \cdot (t_{end} - t_{start})$$

For 2006-2025: $\Delta T_{total} = \beta_1 \cdot 19$

### 3. Statistical Significance

**Coefficient of Determination ($R^2$):**
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

**P-value Interpretation:**
- $p < 0.05$ → Statistically significant trend (*)
- $p \geq 0.05$ → No significant trend

### 4. Mann-Kendall Trend Test
Non-parametric test for monotonic trends:

$$S = \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \text{sgn}(x_j - x_i)$$

Test statistic $Z$ follows standard normal distribution under null hypothesis (no trend).

## Block 1: Setup and Authentication

In [ ]:
# =============================================================================
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# Core libraries
import os
import sys
import json
import time
import re
import glob
import calendar
import warnings
import traceback
import requests
import shutil
from datetime import datetime, timedelta

# Data processing
import numpy as np
import pandas as pd

# Geospatial libraries
import ee                             # Google Earth Engine API
import geemap                         # Interactive mapping with Earth Engine
import geopandas as gpd               # Geospatial data handling
import rasterio                       # Raster data I/O
from rasterio.plot import show        # Plotting raster data
import pyproj                         # Projections and transformations
# Ensure Point is imported for LST extraction
from shapely.geometry import Point

# Statistical analysis
from scipy import stats               # Statistical functions
from scipy.optimize import curve_fit  # Curve fitting
# Use np.inf for infinity, do not import inf from scipy

# Parallel processing with Dask
import dask
from dask.distributed import Client, progress, wait
from dask import delayed
import multiprocessing
from tqdm.notebook import tqdm        # Progress bars

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns

# Fix for libpysal import error with scipy.inf
# Monkey patch scipy.inf before libpysal is imported
import scipy
if not hasattr(scipy, 'inf'):
    import numpy as np
    scipy.inf = np.inf
    
# Spatial statistics (for hotspot analysis)
import libpysal
from libpysal.weights import Queen, KNN
from esda.getisord import G_Local
import pymannkendall

# Import project configuration and custom modules
sys.path.insert(0, os.path.abspath('../'))
import config
from src import gee_utils, temporal_analysis, visualization

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')

print("✅ Libraries imported successfully")
print(f"📁 Project root: {config.PROJECT_ROOT}")
print(f"🗂️ Data directory: {config.DATA_DIR}")

In [ ]:
# =============================================================================
# INITIALIZE GOOGLE EARTH ENGINE
# =============================================================================

print("="*70)
print("INITIALIZING GOOGLE EARTH ENGINE")
print("="*70)

# Initialize GEE using modularized function
gee_available = gee_utils.initialize_gee(
    config.GEE_CONFIG['service_json'],
    config.GEE_CONFIG['project_id']
)

if gee_available:
    print("✅ Google Earth Engine initialized successfully")
    print(f"🔑 Project ID: {config.GEE_CONFIG['project_id']}")
else:
    print("❌ Google Earth Engine initialization failed")
    print("⚠️ Some features will be limited")

# Store GEE availability for later checks
GEE_AVAILABLE = gee_available

print("="*70)

In [ ]:
# Initialize Dask for parallel processing
# Determine number of workers based on CPU cores (leave 1 core for system operations)
n_workers = max(1, multiprocessing.cpu_count() - 1)
n_threads = 2  # Threads per worker

print(f"Setting up Dask with {n_workers} workers and {n_threads} threads per worker")

# Initialize Dask client for parallel processing
client = Client(n_workers=n_workers, threads_per_worker=n_threads)
print(f"Dask dashboard available at: {client.dashboard_link}")

# Function to print Dask cluster information
def print_dask_info():
    info = client.scheduler_info()
    print(f"Dask Cluster Information:")
    print(f"  Workers: {len(info['workers'])}")
    print(f"  Total Threads: {sum(w['nthreads'] for w in info['workers'].values())}")
    print(f"  Total Memory: {sum(w['memory_limit'] for w in info['workers'].values()) / 1e9:.2f} GB")

# Print Dask information
print_dask_info()

## Block 2: Load Configuration

All settings are centralized in [`config.py`](../config.py). This cell creates local references for backward compatibility.

**Key Configuration Parameters:**
- `LST_CONFIG['lst_time']` → 'day', 'night', or 'both'
- `TEMPORAL_CONFIG` → Date ranges (2006-2025), aggregation methods
- `PROCESSING_OPTIONS` → QA tiers (Strict/Moderate), Dask workers
- `VISUALIZATION_CONFIG` → Color schemes, temperature ranges

In [ ]:
# =============================================================================
# LOAD PROJECT CONFIGURATION
# =============================================================================

print("="*70)
print("LOADING PROJECT CONFIGURATION")
print("="*70)

# All configuration is now centralized in config.py
# Users should modify config.py to change behavior

# Create local references for backward compatibility with existing code
# These are REFERENCES, not copies - they point to config.py
PROJECT_CONFIG = {
    'study_year': str(config.PROJECT_SETTINGS['study_year']),
    'start_date': config.TEMPORAL_CONFIG['start_date'],
    'end_date': config.TEMPORAL_CONFIG['end_date'],
    'data_dir': str(config.DATA_DIR),
    'output_dir': str(config.OUTPUT_DATA_PATH),
    'aoi_results_dir': str(config.get_output_dir('01_aoi_delineation')),
    'lst_output_dir': str(config.get_output_dir('03_trend_analysis_monthly_annual')),
    'aoi_pattern': config.PROJECT_SETTINGS['aoi_pattern'],
    'daily_stats_subdir': config.PROJECT_SETTINGS['daily_stats_subdir'],
    'hot_pct': config.PROJECT_SETTINGS['hot_pct']
}

LST_CONFIG = config.LST_CONFIG
TEMPORAL_CONFIG = config.TEMPORAL_CONFIG
VISUALIZATION_CONFIG = config.VISUALIZATION_CONFIG
PROCESSING_OPTIONS = config.PROCESSING_OPTIONS

# Quick validation of LST time selection
valid_lst_times = ['day', 'night', 'both']
if LST_CONFIG['lst_time'] not in valid_lst_times:
    raise ValueError(f"Invalid lst_time in config.py: {LST_CONFIG['lst_time']}. Must be one of {valid_lst_times}")

print(f"✅ Configuration loaded from config.py")
print(f"📅 Time period: {PROJECT_CONFIG['start_date']} to {PROJECT_CONFIG['end_date']}")
print(f"🌡️ LST time mode: {LST_CONFIG['lst_time']}")
print(f"🎯 QA levels to use: {LST_CONFIG['qa_levels_to_use']}")
print(f"📊 Primary collection: {LST_CONFIG['primary_collection']}")
print(f"🔧 Processing options: {PROCESSING_OPTIONS['qa_filtering']} QA filtering")
print("\n⚠️ NOTE: To modify settings, edit config.py and restart the script.")
print("="*70)

## Block 3: Load AOI Data

Auto-loads city boundaries from **Notebook 01** outputs:
- Reads AOI GeoJSON files from `data/outputs/01_aoi_delineation/City_*/`
- Extracts bounding boxes for LST extraction
- Validates geometry and CRS compatibility

In [ ]:
# =============================================================================
# LOAD AOI DATA
# =============================================================================

print("="*70)
print("LOADING AOI BOUNDARIES")
print("="*70)

# Use modularized function to load AOI boundaries
aoi_data = gee_utils.load_aoi_boundaries(
    aoi_results_dir=PROJECT_CONFIG['aoi_results_dir']
)

print(f"\n✅ Loaded AOI data for {len(aoi_data)} cities:")
for city in sorted(aoi_data.keys()):
    bounds = aoi_data[city]['bounds']
    print(f"  📍 {city}: Bounds [{bounds[0]:.3f}, {bounds[1]:.3f}, {bounds[2]:.3f}, {bounds[3]:.3f}]")
    
print("="*70)

## Block 4: LST Extraction with Hierarchical QA Filtering

**Parallel Processing Strategy:**
- Dask workers process multiple cities simultaneously
- Dynamic resource allocation based on CPU cores
- Real-time progress tracking with `tqdm`

**Hierarchical Quality Assurance (QA):**

MODIS LST products include QA bands with error estimates:

| QA Level | Bits | Error Threshold | Description |
|----------|------|-----------------|-------------|
| **Strict** | 00 | ≤ 1K | Highest quality pixels |
| **Moderate** | 01 | ≤ 2K | Medium quality pixels |
| ~~Loose~~ | ~~10~~ | ~~≤ 3K~~ | Not used in this study |

**Fallback Logic:**
1. Extract LST using Strict QA mask
2. If insufficient data → Apply Moderate QA mask
3. Merge results with Strict taking precedence

**Output Products:**
- Monthly climatology GeoTIFFs (12 months)
- Yearly LST GeoTIFFs (2006-2025)
- QA composition maps (percentage of Strict/Moderate pixels)

**Processing Time:** ~5-15 minutes per city (depends on AOI size and time range)

In [ ]:
# Function to load MODIS LST collection with hierarchical quality filtering
def load_modis_collection_with_hierarchical_qa(city_name, geometry, lst_time):
    """
    Load and prepare MODIS LST collection for a city with hierarchical quality filtering
    
    This function implements hierarchical QA filtering for either day or night LST:
    1. First attempt to use strict QA (highest quality pixels)
    2. Where strict QA fails, use moderate QA (if enabled)
    3. Where moderate QA fails, use loose QA (if enabled)
    4. Track which QA tier was used for each pixel
    
    Parameters:
    -----------
    city_name : str
        Name of the city
    geometry : ee.Geometry
        Earth Engine geometry of the city's AOI
    lst_time : str
        'day' or 'night' to specify which LST band to process
        
    Returns:
    --------
    tuple
        (processed_collection, image_count, monthly_counts, yearly_counts, monthly_yearly_counts)
    """
    start_time = time.time()
    try:
        # Get collection configuration
        collection_config = LST_CONFIG['collections'][LST_CONFIG['primary_collection']]
        
        # Select the band based on user preference (day or night)
        selected_band = collection_config[lst_time + '_band']
        selected_qa_band = collection_config[lst_time + '_qa_band']
        
        # Get the QA levels to use from the configuration
        qa_levels_to_use = LST_CONFIG['qa_levels_to_use']
        
        print(f"  🛰️  Loading {collection_config['description']} {lst_time.title()}-time for {city_name}")
        print(f"  📊 Using LST band: {selected_band}")
        print(f"  🔍 Using QA band: {selected_qa_band}")
        print(f"  🔍 Enabled QA levels: {', '.join([level.title() for level in qa_levels_to_use])}")
        
        # Load collection
        collection = ee.ImageCollection(collection_config['id']) \
            .filterDate(PROJECT_CONFIG['start_date'], PROJECT_CONFIG['end_date']) \
            .filterBounds(geometry)
        
        # Check total images before any filtering
        total_images = collection.size().getInfo()
        print(f"    📊 Total images in collection: {total_images}")
        
        # Apply hierarchical QA filtering and scaling to Celsius
        def apply_hierarchical_qa(image):
            # Extract LST and QA bands
            lst = image.select(selected_band)
            qa = image.select(selected_qa_band)
            
            # Convert LST to Celsius (scale * 0.02 - 273.15)
            lst_celsius = lst.multiply(collection_config['scale_factor']).add(collection_config['offset'])
            
            # Create masks for each quality tier based on QA bits 0-1
            # MODIS LST QA bits 0-1 interpretation:
            # 00 = LST produced, good quality (error <= 1K) - STRICT
            # 01 = LST produced, other quality (error <= 2K) - MODERATE
            # 10 = LST produced, not good quality (error <= 3K) - LOOSE
            # 11 = LST not produced due to cloud effects
            
            # Extract quality tiers from config
            qa_hierarchy = LST_CONFIG['quality_hierarchy']
            
            # Initialize variables for masks and LST data by quality level
            quality_masks = {}
            quality_lst = {}
            
            # Create masks for each enabled quality tier
            for tier in qa_hierarchy.keys():
                if tier in qa_levels_to_use:
                    quality_masks[tier] = qa.bitwiseAnd(3).eq(qa_hierarchy[tier]['qa_bits'])
                    quality_lst[tier] = lst_celsius.updateMask(quality_masks[tier])
            
            # Create a quality source band
            # Initialize with zeros
            qa_source = ee.Image(0).rename('QA_Source')
            
            # Fill with quality levels by priority (higher priority overwrites lower)
            # Sort the enabled quality levels by priority
            sorted_qa_levels = sorted(
                [tier for tier in qa_levels_to_use], 
                key=lambda t: qa_hierarchy[t]['priority']
            )
            
            # Fill QA source from lowest to highest priority
            for i, tier in enumerate(sorted_qa_levels):
                qa_level_value = qa_hierarchy[tier]['priority']
                qa_source = qa_source.where(quality_masks[tier], qa_level_value)
            
            # Create a hierarchical LST band
            # Start with the lowest priority quality
            lst_hierarchical = None
            
            # Sort tiers by priority (highest number = lowest priority)
            for i, tier in enumerate(sorted_qa_levels):
                if i == 0:  # First tier (lowest priority in the enabled list)
                    lst_hierarchical = quality_lst[tier]
                else:
                    # Blend in higher priority data
                    lst_hierarchical = lst_hierarchical.blend(quality_lst[tier])
            
            # If no LST data is available (no enabled quality tiers had data)
            if lst_hierarchical is None:
                # Return an empty image with the same properties
                return ee.Image([]).copyProperties(image, ['system:time_start'])
            
            # Rename to standard 'LST' name
            lst_hierarchical = lst_hierarchical.rename('LST')
            
            # Apply reasonable temperature bounds (5°C to 70°C)
            valid_temp = lst_hierarchical.gt(5).And(lst_hierarchical.lt(70))
            lst_hierarchical = lst_hierarchical.updateMask(valid_temp)
            
            # Update the QA source mask to match the final LST mask
            qa_source = qa_source.updateMask(valid_temp)
            
            # Create output image with LST and QA source bands
            return ee.Image([lst_hierarchical, qa_source]).copyProperties(image, ['system:time_start'])
        
        # Apply hierarchical QA processing to the collection
        processed_collection = collection.map(apply_hierarchical_qa)
        
        # Get final image count after filtering
        image_count = processed_collection.size().getInfo()
        print(f"    📊 Processed images after filtering: {image_count}")
        
        # Calculate frequency of images by month and year
        try:
            # Get all dates as milliseconds since epoch
            dates_millis = processed_collection.aggregate_array('system:time_start').getInfo()
            
            # Convert to Python datetime objects
            import datetime
            dates = [datetime.datetime.fromtimestamp(date/1000) for date in dates_millis]
            
            # Count images per month (across all years)
            monthly_counts = {}
            for month in range(1, 13):
                monthly_counts[month] = sum(1 for date in dates if date.month == month)
            
            # Count images per year
            yearly_counts = {}
            for year in TEMPORAL_CONFIG['years']:
                yearly_counts[year] = sum(1 for date in dates if date.year == year)
            
            # Count images per month per year
            monthly_yearly_counts = {}
            for year in TEMPORAL_CONFIG['years']:
                monthly_yearly_counts[year] = {}
                for month in range(1, 13):
                    monthly_yearly_counts[year][month] = sum(1 for date in dates 
                                                           if date.year == year and date.month == month)
            
            # Debug some counts
            print(f"    📊 Image frequency statistics computed")
            end_time = time.time()
            duration = end_time - start_time
            print(f"    ⏱️ Collection loading completed in {duration:.2f} seconds")
            
            return processed_collection, image_count, monthly_counts, yearly_counts, monthly_yearly_counts
            
        except Exception as e:
            print(f"    ⚠️ Could not compute image frequency statistics: {e}")
            end_time = time.time()
            duration = end_time - start_time
            print(f"    ⏱️ Collection loading completed in {duration:.2f} seconds")
            return processed_collection, image_count, None, None, None
        
    except Exception as e:
        print(f"    ❌ Error loading collection for {city_name}: {e}")
        traceback_str = traceback.format_exc()
        print(f"    📋 Traceback: {traceback_str}")
        end_time = time.time()
        duration = end_time - start_time
        print(f"    ⏱️ Collection loading attempted for {duration:.2f} seconds")
        return None, 0, None, None, None

In [ ]:
# Function to compute composite and export as GeoTIFF with QA tracking
def compute_and_export_lst_geotiff_with_qa(collection, geometry, city_name, city_dir, period_type, period_value, lst_time):
    """
    Compute LST composite using median method and export as GeoTIFF, including QA source information
    
    Parameters:
    -----------
    collection : ee.ImageCollection
        The processed Earth Engine image collection
    geometry : ee.Geometry
        Earth Engine geometry of the city's AOI
    city_name : str
        Name of the city
    city_dir : str
        Directory of the city (e.g., "City_Lagos")
    period_type : str
        'yearly' or 'monthly_climatology'
    period_value : int or str
        Year (e.g., 2020) or month number (1-12)
    lst_time : str
        'day' or 'night' to specify which LST band was processed
        
    Returns:
    --------
    dict or None
        Information about the processed data and download status
    """
    start_time = time.time()
    try:
        # Create date filter based on period type
        if period_type == 'yearly':
            start_date = f"{period_value}-01-01"
            end_date = f"{period_value}-12-31"
            period_name = f"{period_value}"
            file_name = f"{city_name}_LST_{lst_time.title()}_Yearly_{period_value}"
            qa_file_name = f"{city_name}_QA_Source_{lst_time.title()}_Yearly_{period_value}"
        elif period_type == 'monthly_climatology':
            # For monthly climatology, we filter by month across all study years
            month = period_value
            # Create a filter for this month across all years
            month_filter = ee.Filter.calendarRange(month, month, 'month')
            
            period_name = f"Month_{month:02d}_AllYears"
            file_name = f"{city_name}_LST_{lst_time.title()}_MonthlyClimatology_{month:02d}"
            qa_file_name = f"{city_name}_QA_Source_{lst_time.title()}_MonthlyClimatology_{month:02d}"
            
            # Special case for climatology: filter the collection by month across all years
            period_collection = collection.filter(month_filter)
            period_count = period_collection.size().getInfo()
            
            print(f"      🔍 Filtering for monthly climatology: Month {month:02d} across all years")
            print(f"      📊 Images found for monthly climatology: {period_count}")
            
            if period_count == 0:
                print(f"      ⚠️  No images available for this monthly climatology")
                end_time = time.time()
                duration = end_time - start_time
                print(f"      ⏱️ Processing attempted for {duration:.2f} seconds")
                return None
            
            # Skip to the composite computation for this special case
            start_date = None
            end_date = None
        else:
            print(f"      ❌ Unknown period type: {period_type}")
            end_time = time.time()
            duration = end_time - start_time
            print(f"      ⏱️ Processing attempted for {duration:.2f} seconds")
            return None
        
        # For yearly periods, filter by date
        if period_type == 'yearly':
            print(f"      🔍 Filtering for period: {start_date} to {end_date}")
            
            # Filter collection for the period
            period_collection = collection.filterDate(start_date, end_date)
            period_count = period_collection.size().getInfo()
            
            print(f"      📊 Images found for period: {period_count}")
            
            if period_count == 0:
                print(f"      ⚠️  No images available for this period")
                end_time = time.time()
                duration = end_time - start_time
                print(f"      ⏱️ Processing attempted for {duration:.2f} seconds")
                return None
        
        # Compute median composite for LST
        lst_composite = period_collection.select('LST').median().rename('LST')
        
        # Compute mode composite for QA_Source to show dominant quality level used
        qa_source_composite = period_collection.select('QA_Source').mode().rename('QA_Source')
        
        print(f"      🔄 Computing composites from {period_count} images")
        
        # Check if the composite has valid pixels
        try:
            # Check if LST has any valid pixels
            pixelCount = lst_composite.reduceRegion(
                reducer=ee.Reducer.count(),
                geometry=geometry,
                scale=LST_CONFIG['scale'],
                maxPixels=1e9
            ).get('LST')
            
            valid_pixels = ee.Number(pixelCount).getInfo()
            print(f"      📊 Valid pixels in composite: {valid_pixels}")
            
            if valid_pixels == 0:
                print(f"      ⚠️ No valid pixels in the composite. Skipping download.")
                end_time = time.time()
                duration = end_time - start_time
                print(f"      ⏱️ Processing completed in {duration:.2f} seconds")
                return {
                    'file_name': file_name,
                    'period': period_name,
                    'image_count': period_count,
                    'status': 'no_valid_pixels'
                }
                
            # Compute statistics on QA source distribution
            qa_stats = qa_source_composite.reduceRegion(
                reducer=ee.Reducer.frequencyHistogram(),
                geometry=geometry,
                scale=LST_CONFIG['scale'],
                maxPixels=1e9
            ).get('QA_Source')
            
            qa_stats_dict = ee.Dictionary(qa_stats).getInfo()
            
            # Extract counts for each QA level and map to tier names
            qa_counts = {}
            for tier, settings in LST_CONFIG['quality_hierarchy'].items():
                priority = settings['priority']
                priority_str = str(priority)
                count = qa_stats_dict.get(priority_str, 0)
                qa_counts[tier] = count
            
            # Print QA distribution
            print(f"      📊 QA Source Distribution:")
            total_pixels_with_qa = sum(qa_counts.values())
            
            for tier in LST_CONFIG['qa_levels_to_use']:
                count = qa_counts.get(tier, 0)
                pct = (count / valid_pixels * 100) if valid_pixels > 0 else 0
                print(f"        {tier.title()} (Priority {LST_CONFIG['quality_hierarchy'][tier]['priority']}): {count} pixels ({pct:.2f}%)")
            
        except Exception as e:
            print(f"      ⚠️ Error checking valid pixels or QA statistics: {e}")
            # Continue anyway, as this is just a check
        
        # Create proper directory structure for this city with Day/Night subfolder
        city_lst_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], city_dir, 'LST_Temporal_Analysis')
        # Add the Day or Night subfolder
        lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
        city_lst_subdir = os.path.join(city_lst_dir, lst_subfolder)
        city_geotiff_dir = os.path.join(city_lst_subdir, 'geotiff')
        
        # Create directories if they don't exist
        os.makedirs(city_lst_dir, exist_ok=True)
        os.makedirs(city_lst_subdir, exist_ok=True)
        os.makedirs(city_geotiff_dir, exist_ok=True)
        
        # Set up download paths for LST and QA
        download_path = os.path.join(city_geotiff_dir, f"{file_name}.tif")
        qa_download_path = os.path.join(city_geotiff_dir, f"{qa_file_name}.tif")
        
        # Download results with retry mechanism
        downloads = []
        
        # 1. Download LST GeoTIFF
        try:
            # Get the URL for downloading the LST GeoTIFF
            lst_download_params = {
                'scale': LST_CONFIG['scale'],
                'crs': 'EPSG:4326',
                'region': geometry,
                'format': 'GEO_TIFF',
                'bands': ['LST'],
                'filename': file_name
            }
            
            # Get download URL
            lst_url = lst_composite.getDownloadURL(lst_download_params)
            
            # Download with retry
            max_retries = 3
            retry_count = 0
            lst_success = False
            
            while retry_count < max_retries and not lst_success:
                try:
                    print(f"      📥 Downloading LST GeoTIFF (attempt {retry_count + 1})...")
                    response = requests.get(lst_url, stream=True, timeout=60)
                    
                    if response.status_code == 200:
                        with open(download_path, 'wb') as f:
                            for chunk in response.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        print(f"      ✅ Saved LST GeoTIFF to: {download_path}")
                        lst_success = True
                    else:
                        print(f"      ⚠️ HTTP error {response.status_code} during download")
                        retry_count += 1
                except Exception as e:
                    print(f"      ⚠️ Download error: {e}")
                    retry_count += 1
                    time.sleep(2)  # Brief pause before retry
            
            if not lst_success:
                print(f"      ❌ Failed to download LST GeoTIFF after {max_retries} attempts")
            
            # 2. Download QA GeoTIFF
            # Get the URL for downloading the QA GeoTIFF
            qa_download_params = {
                'scale': LST_CONFIG['scale'],
                'crs': 'EPSG:4326',
                'region': geometry,
                'format': 'GEO_TIFF',
                'bands': ['QA_Source'],
                'filename': qa_file_name
            }
            
            # Get download URL
            qa_url = qa_source_composite.getDownloadURL(qa_download_params)
            
            # Download with retry
            retry_count = 0
            qa_success = False
            
            while retry_count < max_retries and not qa_success:
                try:
                    print(f"      📥 Downloading QA GeoTIFF (attempt {retry_count + 1})...")
                    response = requests.get(qa_url, stream=True, timeout=60)
                    
                    if response.status_code == 200:
                        with open(qa_download_path, 'wb') as f:
                            for chunk in response.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        print(f"      ✅ Saved QA GeoTIFF to: {qa_download_path}")
                        qa_success = True
                    else:
                        print(f"      ⚠️ HTTP error {response.status_code} during QA download")
                        retry_count += 1
                except Exception as e:
                    print(f"      ⚠️ QA download error: {e}")
                    retry_count += 1
                    time.sleep(2)  # Brief pause before retry
            
            if not qa_success:
                print(f"      ❌ Failed to download QA GeoTIFF after {max_retries} attempts")
                
            # Return results
            end_time = time.time()
            duration = end_time - start_time
            print(f"      ⏱️ Processing completed in {duration:.2f} seconds")
            
            if lst_success and qa_success:
                return {
                    'file_name': file_name,
                    'period': period_name,
                    'image_count': period_count,
                    'status': 'downloaded',
                    'lst_path': download_path,
                    'qa_path': qa_download_path,
                    'processing_time': duration
                }
            elif lst_success:
                return {
                    'file_name': file_name,
                    'period': period_name,
                    'image_count': period_count,
                    'status': 'partial',
                    'lst_path': download_path,
                    'processing_time': duration
                }
            else:
                return {
                    'file_name': file_name,
                    'period': period_name,
                    'image_count': period_count,
                    'status': 'failed',
                    'processing_time': duration
                }
            
        except Exception as e:
            print(f"      ❌ Error during download: {e}")
            traceback_str = traceback.format_exc()
            print(f"      📋 Traceback: {traceback_str}")
            end_time = time.time()
            duration = end_time - start_time
            print(f"      ⏱️ Processing completed in {duration:.2f} seconds")
            return None
        
    except Exception as e:
        print(f"      ❌ Error computing composite: {e}")
        traceback_str = traceback.format_exc()
        print(f"      📋 Traceback: {traceback_str}")
        end_time = time.time()
        duration = end_time - start_time
        print(f"      ⏱️ Processing attempted for {duration:.2f} seconds")
        return None

In [ ]:
# FIXED DASK IMPLEMENTATION FOR GEE DOWNLOADS
# This handles Earth Engine initialization in workers

import os
import time
import traceback
import requests
from tqdm.notebook import tqdm
import ee
import multiprocessing
from dask.distributed import Client, as_completed
from dask import delayed

def init_ee_in_worker():
    """Initialize Earth Engine in each Dask worker"""
    import sys
    import os
    from pathlib import Path
    
    try:
        # Check if EE is already initialized
        ee.Number(1).getInfo()
    except:
        # Initialize EE with service account
        # Add project root to path to import config
        project_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
        sys.path.insert(0, str(project_root))
        import config
        
        from google.oauth2 import service_account
        
        credentials = service_account.Credentials.from_service_account_file(
            config.GEE_CONFIG['service_json'],
            scopes=['https://www.googleapis.com/auth/earthengine']
        )
        
        ee.Initialize(credentials=credentials, project=config.GEE_CONFIG['project_id'])

# Create a function for processing a single time period
def process_city_period(city_name, bounds_list, city_dir, period_type, period_value, lst_time, 
                       lst_config, temporal_config, project_config):
    """
    Process and download a single period of LST data for a specific city
    
    Parameters are passed as simple Python objects, not EE objects
    """
    try:
        # Initialize Earth Engine in this worker
        init_ee_in_worker()
        
        start_time = time.time()
        
        # Recreate the EE geometry from bounds
        city_geometry = ee.Geometry.Rectangle(bounds_list)
        
        # Create return object
        result = {
            'city': city_name,
            'period_type': period_type,
            'period_value': period_value,
            'lst_time': lst_time,
            'status': 'pending'
        }
        
        # Setup file names
        if period_type == 'yearly':
            file_name = f"{city_name}_LST_{lst_time.title()}_Yearly_{period_value}"
            qa_file_name = f"{city_name}_QA_Source_{lst_time.title()}_Yearly_{period_value}"
        else:
            file_name = f"{city_name}_LST_{lst_time.title()}_MonthlyClimatology_{period_value:02d}"
            qa_file_name = f"{city_name}_QA_Source_{lst_time.title()}_MonthlyClimatology_{period_value:02d}"
            
        # Create directory structure
        city_lst_dir = os.path.join(project_config['lst_output_dir'], city_dir, 'LST_Temporal_Analysis')
        lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
        city_lst_subdir = os.path.join(city_lst_dir, lst_subfolder)
        city_geotiff_dir = os.path.join(city_lst_subdir, 'geotiff')
        
        os.makedirs(city_geotiff_dir, exist_ok=True)
        
        # Define file paths
        file_path = os.path.join(city_geotiff_dir, f"{file_name}.tif")
        qa_file_path = os.path.join(city_geotiff_dir, f"{qa_file_name}.tif")
        
        # Check if file already exists
        if os.path.exists(file_path):
            result['status'] = 'skipped'
            result['message'] = f"File already exists: {file_path}"
            return result
            
        # Get collection configuration
        collection_config = lst_config['collections'][lst_config['primary_collection']]
        
        # Select bands
        selected_band = collection_config[lst_time + '_band']
        selected_qa_band = collection_config[lst_time + '_qa_band']
        
        # Create collection
        collection = ee.ImageCollection(collection_config['id']) \
            .filterDate(project_config['start_date'], project_config['end_date']) \
            .filterBounds(city_geometry)
        
        # Apply hierarchical QA filtering
        def apply_hierarchical_qa(image):
            lst = image.select(selected_band)
            qa = image.select(selected_qa_band)
            
            # Convert to Celsius
            lst_celsius = lst.multiply(collection_config['scale_factor']).add(collection_config['offset'])
            
            # Create quality masks
            qa_levels = lst_config['qa_levels_to_use']
            qa_hierarchy = lst_config['quality_hierarchy']
            
            quality_masks = {}
            quality_lst = {}
            
            for tier in qa_levels:
                quality_masks[tier] = qa.bitwiseAnd(3).eq(qa_hierarchy[tier]['qa_bits'])
                quality_lst[tier] = lst_celsius.updateMask(quality_masks[tier])
            
            # Create QA source band
            qa_source = ee.Image(0).rename('QA_Source')
            
            # Sort tiers by priority
            sorted_qa_levels = sorted(qa_levels, key=lambda t: qa_hierarchy[t]['priority'])
            
            # Fill QA source
            for tier in sorted_qa_levels:
                qa_level_value = qa_hierarchy[tier]['priority']
                qa_source = qa_source.where(quality_masks[tier], qa_level_value)
            
            # Create hierarchical LST
            lst_hierarchical = None
            for i, tier in enumerate(sorted_qa_levels):
                if i == 0:
                    lst_hierarchical = quality_lst[tier]
                else:
                    lst_hierarchical = lst_hierarchical.blend(quality_lst[tier])
            
            if lst_hierarchical is None:
                return ee.Image([]).copyProperties(image, ['system:time_start'])
            
            lst_hierarchical = lst_hierarchical.rename('LST')
            
            # Apply temperature bounds
            valid_temp = lst_hierarchical.gt(5).And(lst_hierarchical.lt(70))
            lst_hierarchical = lst_hierarchical.updateMask(valid_temp)
            qa_source = qa_source.updateMask(valid_temp)
            
            return ee.Image([lst_hierarchical, qa_source]).copyProperties(image, ['system:time_start'])
        
        # Apply QA filtering
        processed_collection = collection.map(apply_hierarchical_qa)
        
        # Filter by period
        if period_type == 'yearly':
            filtered_collection = processed_collection.filterDate(f"{period_value}-01-01", f"{period_value}-12-31")
        else:
            filtered_collection = processed_collection.filter(ee.Filter.calendarRange(period_value, period_value, 'month'))
            
        # Check image count
        count = filtered_collection.size().getInfo()
        if count == 0:
            result['status'] = 'no_data'
            result['message'] = f"No images found for {city_name} - {period_type}={period_value} ({lst_time})"
            return result
            
        # Compute composites
        lst_composite = filtered_collection.select('LST').median()
        qa_composite = filtered_collection.select('QA_Source').mode()
        
        # Get download URLs
        download_params = {
            'scale': lst_config['scale'],
            'crs': 'EPSG:4326',
            'region': city_geometry,
            'format': 'GEO_TIFF'
        }
        
        lst_url = lst_composite.getDownloadURL(download_params)
        qa_url = qa_composite.getDownloadURL(download_params)
        
        # Download with retry
        max_retries = 3
        
        # Download LST
        for retry in range(max_retries):
            try:
                response = requests.get(lst_url, timeout=60)
                if response.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(response.content)
                    break
            except Exception as e:
                if retry == max_retries - 1:
                    raise e
                time.sleep(2)
        
        # Download QA
        for retry in range(max_retries):
            try:
                response = requests.get(qa_url, timeout=60)
                if response.status_code == 200:
                    with open(qa_file_path, 'wb') as f:
                        f.write(response.content)
                    break
            except Exception as e:
                if retry == max_retries - 1:
                    raise e
                time.sleep(2)
        
        # Success
        duration = time.time() - start_time
        result['status'] = 'downloaded'
        result['file_path'] = file_path
        result['qa_file_path'] = qa_file_path
        result['duration'] = duration
        result['image_count'] = count
        
        return result
        
    except Exception as e:
        return {
            'city': city_name,
            'period_type': period_type,
            'period_value': period_value,
            'lst_time': lst_time,
            'status': 'error',
            'message': str(e),
            'traceback': traceback.format_exc()
        }

# Main execution block
print("\n🚀 Starting Dask-based LST Data Processing")
print("="*80)

# First, ensure all workers have the EE initialization function
client.run(init_ee_in_worker)

# Create task list
tasks = []
lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]

for city_name, aoi_info in aoi_data.items():
    # Convert bounds to list (not numpy array)
    bounds = aoi_info['bounds']
    bounds_list = bounds.tolist() if hasattr(bounds, 'tolist') else list(bounds)
    city_dir = aoi_info['city_dir']
    
    for lst_time in lst_time_options:
        # Yearly tasks
        for year in TEMPORAL_CONFIG['years']:
            tasks.append({
                'city_name': city_name,
                'bounds_list': bounds_list,
                'city_dir': city_dir,
                'period_type': 'yearly',
                'period_value': year,
                'lst_time': lst_time
            })
        
        # Monthly climatology tasks
        for month in TEMPORAL_CONFIG['months']:
            tasks.append({
                'city_name': city_name,
                'bounds_list': bounds_list,
                'city_dir': city_dir,
                'period_type': 'monthly_climatology',
                'period_value': month,
                'lst_time': lst_time
            })

print(f"Created {len(tasks)} tasks for parallel processing")
print(f"Cities: {', '.join(aoi_data.keys())}")
print(f"Years: {TEMPORAL_CONFIG['years'][0]} - {TEMPORAL_CONFIG['years'][-1]}")
print(f"LST Time: {', '.join(lst_time_options)}")

# Create Dask delayed tasks
# Pass configurations as dictionaries to avoid serialization issues
dask_tasks = []
for task in tasks:
    dask_task = delayed(process_city_period)(
        task['city_name'],
        task['bounds_list'],
        task['city_dir'],
        task['period_type'],
        task['period_value'],
        task['lst_time'],
        dict(LST_CONFIG),  # Convert to dict
        dict(TEMPORAL_CONFIG),  # Convert to dict
        dict(PROJECT_CONFIG)  # Convert to dict
    )
    dask_tasks.append(dask_task)

# Submit tasks
print(f"\n📊 Submitting tasks to Dask cluster...")
print(f"Workers: {len(client.scheduler_info()['workers'])}")
print(f"Dashboard: {client.dashboard_link}")
futures = client.compute(dask_tasks)

# Track progress
results = []
errors = []

with tqdm(total=len(tasks), desc="Processing LST data") as pbar:
    for future, result in as_completed(futures, with_results=True):
        results.append(result)
        
        if result['status'] == 'downloaded':
            pbar.set_postfix_str(f"✅ {result['city']} - {result['period_type']}={result['period_value']} ({result['lst_time']})")
        elif result['status'] == 'skipped':
            pbar.set_postfix_str(f"⏭️ Skipped: {result['city']}")
        elif result['status'] == 'error':
            errors.append(result)
            pbar.set_postfix_str(f"❌ Error: {result['city']}")
            
        pbar.update(1)

# Summary
print("\n" + "="*80)
print("🎯 PROCESSING SUMMARY")
print("="*80)

downloaded = [r for r in results if r['status'] == 'downloaded']
skipped = [r for r in results if r['status'] == 'skipped']
no_data = [r for r in results if r['status'] == 'no_data']

print(f"Total tasks: {len(results)}")
print(f"Downloaded: {len(downloaded)}")
print(f"Skipped (already exist): {len(skipped)}")
print(f"No data available: {len(no_data)}")
print(f"Errors: {len(errors)}")

if errors:
    print("\n❌ ERRORS:")
    for error in errors[:5]:  # Show first 5 errors
        print(f"\n{error['city']} - {error['period_type']}={error['period_value']} ({error['lst_time']}):")
        print(f"  {error['message']}")

print("\n✅ Processing complete!")
print(f"Check output directories in: {PROJECT_CONFIG['lst_output_dir']}")

## Block 5: Visualize LST Maps

Generates publication-ready maps in A5 format (5.83" × 8.27"):

**Monthly Climatology Maps:**
- 12 subplots (one per month) in single figure
- Color scale: Perceptually uniform "RdYlBu_r"
- Consistent scale across all months for comparability

**Yearly LST Maps:**
- 20 subplots (2006-2025) in single figure
- Tracks inter-annual temperature variations
- Identifies anomalously warm/cool years

**QA Composition Maps:**
- Visualizes spatial distribution of QA levels
- Color coding: Green (Strict) → Yellow (Moderate) → Red (Poor/No Data)

**Output Formats:** PNG (screen viewing) and PDF (publication-ready)

In [ ]:
print("="*70)
print("🗺️DASK-BASED VISUALIZATION WITH QA STATISTICS")
print("="*70)

import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for parallel processing
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import rasterio
import numpy as np
import glob
import re
from dask import delayed
from dask.distributed import as_completed

# Define A5 size in inches
A5_SIZE_INCHES = (5.83, 8.27)  # Width x Height

# Function to load and prepare GeoTIFF data
def load_geotiff(file_path):
    """Load GeoTIFF file and prepare for plotting"""
    try:
        with rasterio.open(file_path) as src:
            data = src.read(1)
            bounds = src.bounds
            
            # Replace NoData values with NaN
            if src.nodata is not None:
                data = np.where(data == src.nodata, np.nan, data)
            
            # Mask extreme values
            data = np.where((data < -50) | (data > 70), np.nan, data)
            
            return data, bounds, src.transform, src.crs
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return None, None, None, None

# Function to calculate QA statistics
def calculate_qa_stats(qa_file_path):
    """Calculate QA statistics from QA GeoTIFF file"""
    try:
        with rasterio.open(qa_file_path) as src:
            qa_data = src.read(1)
            valid_pixels = np.count_nonzero(~np.isnan(qa_data))
            
            if valid_pixels == 0:
                return "No QA data"
            
            # Count pixels with different QA values (1 = Strict, 2 = Moderate)
            strict_pixels = np.count_nonzero(qa_data == 1)
            strict_pct = (strict_pixels / valid_pixels) * 100
            
            return f"QA: S-{strict_pct:.1f}%"
    except Exception as e:
        return "QA data unavailable"

# Dask function to create monthly climatology map
@delayed
def create_monthly_climatology_map(city_name, city_dir, lst_time, lst_config, vis_config, temporal_config, project_config):
    """Create monthly climatology map for a city and time period"""
    try:
        # Setup directories based on your structure
        city_lst_dir = os.path.join(project_config['lst_output_dir'], city_dir, 'LST_Temporal_Analysis')
        lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
        city_lst_subdir = os.path.join(city_lst_dir, lst_subfolder)
        geotiff_dir = os.path.join(city_lst_subdir, 'geotiff')
        
        if not os.path.exists(geotiff_dir):
            return {'status': 'error', 'message': f'Directory not found: {geotiff_dir}'}
        
        # Find monthly files
        monthly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_MonthlyClimatology_*.tif")
        monthly_files = sorted(glob.glob(monthly_pattern))
        
        if not monthly_files:
            return {'status': 'no_data', 'message': f'No monthly files found for {city_name} {lst_time}'}
        
        # Create figures directory
        figures_dir = os.path.join(city_lst_subdir, 'Figures')
        os.makedirs(figures_dir, exist_ok=True)
        
        # Create figure with tight_layout to fix the MixedModeRenderer error
        fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
        
        # Use a proper tight layout to avoid issues with mixed renderer
        plt.tight_layout()
        
        # Create GridSpec after tight_layout
        gs = GridSpec(4, 3, figure=fig, hspace=0.15, wspace=0.1)
        
        month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                      'July', 'August', 'September', 'October', 'November', 'December']
        
        # Get visualization settings
        min_val = vis_config['min_temp']
        max_val = vis_config['max_temp']
        highlight_temp = vis_config['highlight_temp']
        lst_cmap = LinearSegmentedColormap.from_list(vis_config['colormap_name'], vis_config['colors'])
        
        # Process each month
        im = None
        for file_path in monthly_files:
            # Extract month number
            file_name = os.path.basename(file_path)
            month_match = re.search(r'MonthlyClimatology_(\d+)', file_name)
            
            if not month_match:
                continue
                
            month_num = int(month_match.group(1))
            
            # Calculate grid position
            row = (month_num - 1) // 3
            col = (month_num - 1) % 3
            
            # Create subplot
            ax = fig.add_subplot(gs[row, col])
            
            # Load and plot data
            data, bounds, transform, crs = load_geotiff(file_path)
            
            if data is not None:
                im = ax.imshow(data, cmap=lst_cmap, vmin=min_val, vmax=max_val)
                ax.set_title(month_names[month_num-1], fontsize=8)
                ax.set_xticks([])
                ax.set_yticks([])
                
                # Add QA stats
                qa_file_path = file_path.replace('_LST_', '_QA_Source_')
                if os.path.exists(qa_file_path):
                    qa_stats_str = calculate_qa_stats(qa_file_path)
                    ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom', 
                           transform=ax.transAxes, fontsize=6.5,
                           bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
            else:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(month_names[month_num-1], fontsize=8)
        
        # Force drawing the figure before adding colorbar (fixes renderer issues)
        fig.canvas.draw()
        
        # Add colorbar
        if im is not None:
            cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
            tick_spacing = 5
            ticks = list(range(int(min_val), int(max_val) + 1, tick_spacing))
            if highlight_temp not in ticks:
                ticks.append(highlight_temp)
                ticks.sort()
            
            cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', ticks=ticks)
            cbar_labels = [str(temp) for temp in ticks]
            cbar.ax.set_xticklabels(cbar_labels, fontsize=6)
            labels = cbar.ax.get_xticklabels()
            highlight_index = ticks.index(highlight_temp)
            labels[highlight_index].set_weight('bold')
            cbar.set_label('Temperature (°C)', fontsize=8)
        
        # Add title and metadata
        fig.suptitle(f"{city_name} - {lst_time.title()}-time Monthly LST Climatology", fontsize=10, y=0.95)
        years_range = f"{temporal_config['years'][0]}-{temporal_config['years'][-1]}"
        fig.text(0.5, 0.01, f"Median {lst_time.title()}-time LST values across years {years_range}", 
                ha='center', fontsize=6)
        
        # Draw again to ensure all elements are properly rendered
        fig.canvas.draw()
        
        # Save figure - ensure we call draw before saving
        output_file_pdf = os.path.join(figures_dir, f"{city_name}_{lst_time.title()}_Monthly_LST_Climatology.pdf")
        output_file_png = os.path.join(figures_dir, f"{city_name}_{lst_time.title()}_Monthly_LST_Climatology.png")
        
        # Use explicit renderer for PDF to fix print_pdf errors
        fig.savefig(output_file_pdf, bbox_inches='tight', format='pdf')
        fig.savefig(output_file_png, bbox_inches='tight', dpi=300, format='png')
        plt.close(fig)
        
        return {
            'status': 'success',
            'city': city_name,
            'lst_time': lst_time,
            'type': 'monthly',
            'output_pdf': output_file_pdf,
            'output_png': output_file_png
        }
        
    except Exception as e:
        import traceback
        trace = traceback.format_exc()
        error_msg = f"{str(e)}\n{trace}"
        return {
            'status': 'error',
            'city': city_name,
            'lst_time': lst_time,
            'type': 'monthly',
            'message': error_msg
        }

# Dask function to create yearly LST map
@delayed
def create_yearly_lst_map(city_name, city_dir, lst_time, lst_config, vis_config, temporal_config, project_config):
    """Create yearly LST map for a city and time period"""
    try:
        # Setup directories
        city_lst_dir = os.path.join(project_config['lst_output_dir'], city_dir, 'LST_Temporal_Analysis')
        lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
        city_lst_subdir = os.path.join(city_lst_dir, lst_subfolder)
        geotiff_dir = os.path.join(city_lst_subdir, 'geotiff')
        
        if not os.path.exists(geotiff_dir):
            return {'status': 'error', 'message': f'Directory not found: {geotiff_dir}'}
        
        # Find yearly files
        yearly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_Yearly_*.tif")
        yearly_files = sorted(glob.glob(yearly_pattern))
        
        if not yearly_files:
            return {'status': 'no_data', 'message': f'No yearly files found for {city_name} {lst_time}'}
        
        # Create figures directory
        figures_dir = os.path.join(city_lst_subdir, 'Figures')
        os.makedirs(figures_dir, exist_ok=True)
        
        # Create figure with tight_layout to fix renderer issues
        fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
        plt.tight_layout()
        
        n_rows, n_cols = 5, 4
        gs = GridSpec(n_rows, n_cols, figure=fig, hspace=0.15, wspace=0.1)
        
        # Get visualization settings
        min_val = vis_config['min_temp']
        max_val = vis_config['max_temp']
        highlight_temp = vis_config['highlight_temp']
        lst_cmap = LinearSegmentedColormap.from_list(vis_config['colormap_name'], vis_config['colors'])
        
        # Process each year
        im = None
        for i, file_path in enumerate(yearly_files[:n_rows*n_cols]):  # Limit to grid capacity
            # Extract year
            file_name = os.path.basename(file_path)
            year_match = re.search(r'Yearly_(\d{4})', file_name)
            
            if not year_match:
                continue
                
            year_str = year_match.group(1)
            
            # Calculate grid position
            row = i // n_cols
            col = i % n_cols
            
            # Create subplot
            ax = fig.add_subplot(gs[row, col])
            
            # Load and plot data
            data, bounds, transform, crs = load_geotiff(file_path)
            
            if data is not None:
                im = ax.imshow(data, cmap=lst_cmap, vmin=min_val, vmax=max_val)
                ax.set_title(year_str, fontsize=8)
                ax.set_xticks([])
                ax.set_yticks([])
                
                # Add QA stats
                qa_file_path = file_path.replace('_LST_', '_QA_Source_')
                if os.path.exists(qa_file_path):
                    qa_stats_str = calculate_qa_stats(qa_file_path)
                    ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom', 
                           transform=ax.transAxes, fontsize=6.5,
                           bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
            else:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(year_str, fontsize=8)
        
        # Force drawing the figure before adding colorbar (fixes renderer issues)
        fig.canvas.draw()
        
        # Add colorbar
        if im is not None:
            cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
            tick_spacing = 5
            ticks = list(range(int(min_val), int(max_val) + 1, tick_spacing))
            if highlight_temp not in ticks:
                ticks.append(highlight_temp)
                ticks.sort()
            
            cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', ticks=ticks)
            cbar_labels = [str(temp) for temp in ticks]
            cbar.ax.set_xticklabels(cbar_labels, fontsize=6)
            labels = cbar.ax.get_xticklabels()
            highlight_index = ticks.index(highlight_temp)
            labels[highlight_index].set_weight('bold')
            cbar.set_label('Temperature (°C)', fontsize=8)
        
        # Add title
        fig.suptitle(f"{city_name} - {lst_time.title()}-time Yearly LST Medians", fontsize=10, y=0.95)
        
        # Draw again to ensure all elements are properly rendered
        fig.canvas.draw()
        
        # Save figure with explicit formats
        output_file_pdf = os.path.join(figures_dir, f"{city_name}_{lst_time.title()}_Yearly_LST.pdf")
        output_file_png = os.path.join(figures_dir, f"{city_name}_{lst_time.title()}_Yearly_LST.png")
        
        fig.savefig(output_file_pdf, bbox_inches='tight', format='pdf')
        fig.savefig(output_file_png, bbox_inches='tight', dpi=300, format='png')
        plt.close(fig)
        
        return {
            'status': 'success',
            'city': city_name,
            'lst_time': lst_time,
            'type': 'yearly',
            'output_pdf': output_file_pdf,
            'output_png': output_file_png
        }
        
    except Exception as e:
        import traceback
        trace = traceback.format_exc()
        error_msg = f"{str(e)}\n{trace}"
        return {
            'status': 'error',
            'city': city_name,
            'lst_time': lst_time,
            'type': 'yearly',
            'message': error_msg
        }

# Main execution using Dask
print("\n🚀 Starting Dask-based visualization processing")
print("="*70)

# Create visualization tasks
viz_tasks = []
lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]

for city_name, aoi_info in aoi_data.items():
    city_dir = aoi_info['city_dir']
    
    for lst_time in lst_time_options:
        # Get visualization config for this time
        vis_config = VISUALIZATION_CONFIG[lst_time]
        
        # Create monthly climatology task
        monthly_task = create_monthly_climatology_map(
            city_name, city_dir, lst_time,
            dict(LST_CONFIG), dict(vis_config), 
            dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)
        )
        viz_tasks.append(monthly_task)
        
        # Create yearly map task
        yearly_task = create_yearly_lst_map(
            city_name, city_dir, lst_time,
            dict(LST_CONFIG), dict(vis_config),
            dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)
        )
        viz_tasks.append(yearly_task)

print(f"Created {len(viz_tasks)} visualization tasks")
print(f"Cities: {', '.join(aoi_data.keys())}")
print(f"Time periods: {', '.join(lst_time_options)}")

# Submit tasks to Dask
print(f"\n📊 Submitting visualization tasks to Dask cluster...")
print(f"Workers: {len(client.scheduler_info()['workers'])}")
print(f"Dashboard: {client.dashboard_link}")

futures = client.compute(viz_tasks)

# Track progress
results = []
with tqdm(total=len(viz_tasks), desc="Creating visualizations") as pbar:
    for future, result in as_completed(futures, with_results=True):
        results.append(result)
        
        if result['status'] == 'success':
            viz_type = result['type']
            pbar.set_postfix_str(f"✅ {result['city']} - {result['lst_time']} {viz_type}")
        elif result['status'] == 'error':
            pbar.set_postfix_str(f"❌ Error: {result['city']} - {result['lst_time']} {result['type']}")
        elif result['status'] == 'no_data':
            pbar.set_postfix_str(f"⚠️ No data: {result['city']}")
            
        pbar.update(1)

# Summary
print("\n" + "="*70)
print("🎯 VISUALIZATION SUMMARY")
print("="*70)

successful = [r for r in results if r['status'] == 'success']
errors = [r for r in results if r['status'] == 'error']
no_data = [r for r in results if r['status'] == 'no_data']

print(f"Total tasks: {len(results)}")
print(f"Successful: {len(successful)}")
print(f"No data: {len(no_data)}")
print(f"Errors: {len(errors)}")

# Group by city and type
if successful:
    print("\n✅ SUCCESSFULLY CREATED:")
    for city in sorted(aoi_data.keys()):
        city_results = [r for r in successful if r['city'] == city]
        if city_results:
            print(f"\n{city}:")
            for result in city_results:
                print(f"  - {result['lst_time'].title()}-time {result['type']} map")
                print(f"    PDF: {os.path.basename(result['output_pdf'])}")

if errors:
    print("\n❌ ERRORS:")
    for error in errors[:5]:  # Show first 5 errors
        print(f"\n{error['city']} - {error['lst_time']} {error['type']}:")
        print(f"  {error.get('message', 'Unknown error')[:200]}")  # Limit error message length

print("\n✅ Visualization processing complete!")
print(f"Check figures in each city's LST_Temporal_Analysis/[Day|Night]/Figures directory")
print("="*70)

In [ ]:
print("="*70)
print("🗺️QA MAP VISUALIZATION")
print("="*70)

# Import required color mapping classes
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.gridspec as gridspec
import re

# Define A5 size in inches for consistent figure size
if 'A5_SIZE_INCHES' not in globals():
    A5_SIZE_INCHES = (8.27, 11.69)  # A5 size in inches

def load_qa_geotiff(file_path):
    """Load QA GeoTIFF file and prepare for plotting"""
    try:
        with rasterio.open(file_path) as src:
            # Read the data
            data = src.read(1)
            
            # Get metadata
            bounds = src.bounds
            
            # Replace NoData values with NaN
            if src.nodata is not None:
                data = np.where(data == src.nodata, np.nan, data)
            
            return data, bounds, src.transform, src.crs
    except Exception as e:
        print(f"❌ Error loading QA file {file_path}: {e}")
        return None, None, None, None

# Function to calculate detailed QA statistics from QA GeoTIFF
def calculate_detailed_qa_stats(qa_data):
    """Calculate detailed QA statistics from QA data"""
    try:
        # Count total valid pixels
        valid_pixels = np.count_nonzero(~np.isnan(qa_data))
        
        if valid_pixels == 0:
            return "No QA data"
        
        # Count pixels with different QA values based on quality hierarchy
        qa_counts = {}
        for tier, config in LST_CONFIG['quality_hierarchy'].items():
            if tier in LST_CONFIG['qa_levels_to_use']:
                priority_value = config['priority']
                qa_counts[tier] = np.count_nonzero(qa_data == priority_value)
        
        # Calculate percentages and build stats string
        qa_stats_parts = []
        for tier, count in qa_counts.items():
            pct = (count / valid_pixels) * 100
            # Use first letter of tier name and percentage
            short_name = tier[0].upper()
            qa_stats_parts.append(f"{short_name}:{pct:.1f}%")
            
        qa_stats_str = " | ".join(qa_stats_parts)
        return qa_stats_str
    except Exception as e:
        print(f"❌ Error calculating QA stats: {e}")
        return "QA data unavailable"

# Create QA colormap based on configuration
qa_colors = []
qa_labels = []
qa_values = []

# Build colors and labels from configuration
if 'LST_CONFIG' in globals() and LST_CONFIG.get('qa_levels_to_use'):
    for tier in sorted(LST_CONFIG['qa_levels_to_use'], key=lambda t: LST_CONFIG['quality_hierarchy'][t]['priority']):
        tier_config = LST_CONFIG['quality_hierarchy'][tier]
        qa_colors.append(tier_config['color'])
        # Defensive: only split if '≤' is present
        if '≤' in tier_config['description']:
            qa_labels.append(f"{tier.capitalize()} (≤{tier_config['description'].split('≤')[1].strip()})")
        else:
            qa_labels.append(tier.capitalize())
        qa_values.append(tier_config['priority'])

if qa_colors and qa_values:
    # Create custom colormap for QA maps
    qa_cmap = ListedColormap(qa_colors)
    qa_bounds = [qa_values[0] - 0.5] + [v + 0.5 for v in qa_values]
    qa_norm = BoundaryNorm(qa_bounds, len(qa_colors))
else:
    print("❌ QA configuration is missing or empty. Please check LST_CONFIG['qa_levels_to_use'] and LST_CONFIG['quality_hierarchy'].")
    qa_cmap = None
    qa_bounds = None
    qa_norm = None

# Process one city at a time for QA maps
for city_name, aoi_info in aoi_data.items():
    print(f"\n🏙️ Processing QA visualizations for {city_name}")
    city_dir = aoi_info['city_dir']
    
    # Process both day and night
    for lst_time in lst_time_options:
        print(f"\n  🕒 Processing {lst_time}-time QA visualizations")
        
        # Get the appropriate subfolder based on time of day
        subdir = 'Day' if lst_time == 'day' else 'Night'
        
        # Use the correct directory structure based on the observed file pattern
        # ../outputs/City_*/LST_Temporal_Analysis/{Day/Night}/geotiff/
        geotiff_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], city_dir, 'LST_Temporal_Analysis', subdir, 'geotiff')
        
        if not os.path.exists(geotiff_dir):
            print(f"  ❌ Geotiff directory not found: {geotiff_dir}")
            continue
        
        print(f"  ✅ Found geotiff directory: {geotiff_dir}")
        
        # Create Figures directory for QA maps (based on project structure)
        figures_dir = os.path.join(os.path.dirname(geotiff_dir), 'Figures')
        os.makedirs(figures_dir, exist_ok=True)
        
        qa_maps_dir = os.path.join(figures_dir, 'QA_Maps')
        os.makedirs(qa_maps_dir, exist_ok=True)
        
        # === MONTHLY QA CLIMATOLOGY MAP ===
        
        # Find monthly QA files using the actual naming pattern
        # Pattern: {city_name}_QA_Source_{day/night}_MonthlyClimatology_{month}.tif
        monthly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_MonthlyClimatology_*.tif")
        
        # Alternative pattern observed in the directory
        alt_monthly_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_month_*.tif")
        
        # Try both patterns
        monthly_qa_files = sorted(glob.glob(monthly_qa_pattern))
        if not monthly_qa_files:
            monthly_qa_files = sorted(glob.glob(alt_monthly_pattern))
        
        # If still not found, try climatology pattern
        if not monthly_qa_files:
            monthly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_month_*_QA_Source_{lst_time}_climatology.tif")
            monthly_qa_files = sorted(glob.glob(monthly_qa_pattern))
        
        print(f"  🔍 Searching for monthly QA files")
        
        if not monthly_qa_files:
            print(f"  ❌ No monthly QA climatology files found for {city_name} ({lst_time}-time)")
        else:
            print(f"  ✅ Found {len(monthly_qa_files)} monthly QA climatology files")
            
            try:
                print(f"  📊 Creating monthly QA climatology map...")
                
                # Create figure with 4x3 grid (one for each month)
                fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
                gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.15, wspace=0.1)  # Reduced spacing
                
                # Define month names
                month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                               'July', 'August', 'September', 'October', 'November', 'December']
                
                # Create plots
                for file_path in monthly_qa_files:
                    # Extract month number from filename
                    file_name = os.path.basename(file_path)
                    
                    # Try different patterns to extract month number
                    month_match = re.search(r'MonthlyClimatology_(\d+)', file_name) or \
                                  re.search(r'month_(\d+)', file_name) or \
                                  re.search(r'_(\d{2})\.tif', file_name)
                    
                    if not month_match:
                        print(f"  ⚠️ Warning: Could not extract month from filename: {file_name}")
                        continue
                        
                    month_str = month_match.group(1)
                    month_num = int(month_str)
                    
                    # Calculate grid position
                    row = (month_num - 1) // 3
                    col = (month_num - 1) % 3
                    
                    # Create subplot
                    ax = fig.add_subplot(gs[row, col])
                    
                    # Load and plot QA data
                    qa_data, bounds, transform, crs = load_qa_geotiff(file_path)
                    
                    if qa_data is not None:
                        im = ax.imshow(qa_data, cmap=qa_cmap, norm=qa_norm)
                        ax.set_title(month_names[month_num-1], fontsize=8)
                        ax.set_xticks([])
                        ax.set_yticks([])
                        
                        # Calculate detailed QA stats
                        qa_stats_str = calculate_detailed_qa_stats(qa_data)
                        
                        # Add QA stats to bottom-right corner with white background
                        ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom', 
                                transform=ax.transAxes, fontsize=7.5,
                                bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
                    else:
                        ax.text(0.5, 0.5, 'No QA Data', ha='center', va='center', transform=ax.transAxes)
                        ax.set_title(month_names[month_num-1], fontsize=8)
                
                # Add custom colorbar at the bottom
                cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
                cbar = plt.colorbar(
                    plt.cm.ScalarMappable(cmap=qa_cmap, norm=qa_norm),
                    cax=cbar_ax,
                    orientation='horizontal',
                    ticks=qa_values
                )
                cbar.ax.set_xticklabels(qa_labels, fontsize=6)
                cbar.set_label('QA Level', fontsize=8)
                
                # Add title and metadata
                fig.suptitle(f"{city_name} - {lst_time.title()}-time Monthly QA Climatology", fontsize=10, y=0.95)
                years_range = f"{TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]}"
                plt.figtext(0.5, 0.01, f"Data period: {years_range}", ha='center', fontsize=6)
                
                # Save figure
                output_file_pdf = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Monthly_QA_Climatology.pdf")
                output_file_png = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Monthly_QA_Climatology.png")
                plt.savefig(output_file_pdf, bbox_inches='tight')
                plt.savefig(output_file_png, bbox_inches='tight', dpi=300)
                plt.close()
                
                print(f"  ✅ Saved monthly QA climatology map to {output_file_pdf}")
                print(f"    and {output_file_png}")
            
            except Exception as e:
                print(f"  ❌ Error creating monthly QA maps for {city_name} ({lst_time}-time): {e}")
                traceback_str = traceback.format_exc()
                print(f"  📋 Traceback: {traceback_str}")
        
        # === YEARLY QA MAP ===
        
        # Find yearly QA files using the actual naming pattern
        # Pattern: {city_name}_QA_Source_{day/night}_Yearly_{year}.tif
        yearly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_Yearly_*.tif")
        
        # Alternative pattern if the first one doesn't match
        alt_yearly_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_yearly_*.tif")
        
        # Try both patterns
        yearly_qa_files = sorted(glob.glob(yearly_qa_pattern))
        if not yearly_qa_files:
            yearly_qa_files = sorted(glob.glob(alt_yearly_pattern))
            
        # If still not found, try the pattern from the implementation
        if not yearly_qa_files:
            yearly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_*_QA_Source_{lst_time}_yearly.tif")
            yearly_qa_files = sorted(glob.glob(yearly_qa_pattern))
            
        print(f"\n  🔍 Searching for yearly QA files")
        
        if not yearly_qa_files:
            print(f"  ❌ No yearly QA files found for {city_name} ({lst_time}-time)")
            continue
        
        print(f"  ✅ Found {len(yearly_qa_files)} yearly QA files")
        
        try:
            # Create figure for yearly QA maps - adjust grid based on number of years
            n_years = len(yearly_qa_files)
            
            # Determine grid layout based on number of years
            if n_years <= 12:
                n_rows = 3
                n_cols = 4
            elif n_years <= 16:
                n_rows = 4
                n_cols = 4
            elif n_years <= 20:
                n_rows = 5
                n_cols = 4
            else:
                n_rows = 5
                n_cols = 5
                
            fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
            gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.15, wspace=0.1)
            
            # Create plots
            for i, file_path in enumerate(yearly_qa_files):
                # Extract year from filename
                file_name = os.path.basename(file_path)
                
                # Try different patterns to extract year
                year_match = re.search(r'Yearly_(\d{4})', file_name) or \
                             re.search(r'yearly_(\d{4})', file_name) or \
                             re.search(r'_(\d{4})_', file_name)
                
                if not year_match:
                    print(f"  ⚠️ Warning: Could not extract year from filename: {file_name}")
                    continue
                    
                year_str = year_match.group(1)
                
                # Calculate grid position
                row = i // n_cols
                col = i % n_cols
                
                # Skip if we exceed the grid capacity
                if row >= n_rows:
                    print(f"  ⚠️ Warning: More years ({n_years}) than grid capacity ({n_rows*n_cols})")
                    print(f"      Skipping year {year_str} and subsequent years")
                    break
                
                # Create subplot
                ax = fig.add_subplot(gs[row, col])
                
                # Load and plot QA data
                qa_data, bounds, transform, crs = load_qa_geotiff(file_path)
                
                if qa_data is not None:
                    im = ax.imshow(qa_data, cmap=qa_cmap, norm=qa_norm)
                    ax.set_title(year_str, fontsize=8)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    
                    # Calculate detailed QA stats
                    qa_stats_str = calculate_detailed_qa_stats(qa_data)
                    
                    # Add QA stats to bottom-right corner with white background
                    ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom', 
                            transform=ax.transAxes, fontsize=7.5,
                            bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
                else:
                    ax.text(0.5, 0.5, 'No QA Data', ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(year_str, fontsize=8)
            
            # Add custom colorbar at the bottom
            cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
            cbar = plt.colorbar(
                plt.cm.ScalarMappable(cmap=qa_cmap, norm=qa_norm),
                cax=cbar_ax,
                orientation='horizontal',
                ticks=qa_values
            )
            cbar.ax.set_xticklabels(qa_labels, fontsize=6)
            cbar.set_label('QA Level', fontsize=8)
            
            # Add title and metadata
            fig.suptitle(f"{city_name} - {lst_time.title()}-time Yearly QA Maps", fontsize=10, y=0.95)
                        
            # Save figure
            output_file_pdf = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Yearly_QA.pdf")
            output_file_png = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Yearly_QA.png")
            plt.savefig(output_file_pdf, bbox_inches='tight')
            plt.savefig(output_file_png, bbox_inches='tight', dpi=300)
            plt.close()
            
            print(f"  ✅ Saved yearly QA map to {output_file_pdf}")
            print(f"    and {output_file_png}")
            
        except Exception as e:
            print(f"  ❌ Error creating yearly QA maps for {city_name} ({lst_time}-time): {e}")
            traceback_str = traceback.format_exc()
            print(f"  📋 Traceback: {traceback_str}")

print("\n" + "="*70)
print("🎯 QA MAP VISUALIZATION COMPLETE")
print("="*70)
print("Check each city's QA maps in the Figures/QA_Maps directory")
print("="*70)

## Block 6: Compute LST Statistics

Extracts comprehensive statistics from GeoTIFF files:

**Data Filtering:**
- Excludes LST < 10°C (likely water bodies, cloud artifacts, no-data)
- Retains only valid land surface temperatures

**Computed Metrics:**

For each city, month/year, and time period (day/night):

| Metric | Formula | Description |
|--------|---------|-------------|
| Mean | $\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i$ | Average temperature |
| Std Dev | $\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$ | Temperature variability |
| Min/Max | - | Temperature extremes |
| Range | $R = x_{max} - x_{min}$ | Thermal amplitude |
| Median | $\tilde{x}$ | 50th percentile |

**Output Files:**
- Per-city CSV: `City_*/LST_Temporal_Analysis/[Day/Night]/Statistics/`
- Combined CSV: `Cross_City_Statistics/[monthly/yearly]_LST_statistics_[day/night].csv`

In [ ]:
print("="*70)
print("BLOCK 6: LST SUMMARY WITH YEARLY STATISTICS")
print("="*70)

# Start timing for enhanced statistics
final_stats_start_time = time.time()

# Define the main statistics directory - use a new "Stats" folder
main_stats_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats')
os.makedirs(main_stats_dir, exist_ok=True)
print(f"\nMain statistics directory: {main_stats_dir}")

# Create a better structured summary file with more statistics
print("\n📋 Creating enhanced summary report with yearly statistics and additional metrics")

try:
    # Path for our target statistics file in the new location
    target_stats_file = os.path.join(main_stats_dir, "All_Cities_LST_Statistics.csv")
    
    # First, check possible locations for the statistics file
    possible_source_files = [
        # Check in LST_Temporal_Analysis_Dask directory
        os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', "All_Cities_LST_Statistics.csv"),
        # Check in each city's directory
        os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', "City_LST_Statistics.csv"),
    ]
    
    # Variable to store if we found and copied an existing file
    file_found = False
    
    # Check if any of these files exist and copy the first one found
    for source_file in possible_source_files:
        if os.path.exists(source_file):
            import shutil
            shutil.copy2(source_file, target_stats_file)
            print(f"✅ Copied statistics file from {source_file} to: {target_stats_file}")
            file_found = True
            break
    
    # If we didn't find any existing files, collect and merge individual city statistics files
    if not file_found:
        print("⚠️ No existing statistics summary file found. Computing statistics from GeoTIFF files...")
        
        # Compute statistics directly from GeoTIFF files
        all_city_stats = []
        
        for city_name, aoi_info in aoi_data.items():
            city_dir = aoi_info['city_dir']
            print(f"\n📊 Computing statistics for {city_name}...")
            
            # Process both day and night
            for lst_time in lst_time_options:
                lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
                city_lst_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], city_dir, 'LST_Temporal_Analysis', lst_subfolder)
                geotiff_dir = os.path.join(city_lst_dir, 'geotiff')
                
                if not os.path.exists(geotiff_dir):
                    print(f"  ⚠️ Directory not found: {geotiff_dir}")
                    continue
                
                # Process yearly files
                yearly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_Yearly_*.tif")
                yearly_files = sorted(glob.glob(yearly_pattern))
                
                for file_path in yearly_files:
                    try:
                        year_match = re.search(r'Yearly_(\d{4})', file_path)
                        if not year_match:
                            continue
                        year = int(year_match.group(1))
                        
                        # Read GeoTIFF and compute statistics
                        with rasterio.open(file_path) as src:
                            data = src.read(1)
                            if src.nodata is not None:
                                data = np.where(data == src.nodata, np.nan, data)
                            # Apply realistic temperature bounds for Nigerian cities
                            data = np.where((data < 10) | (data > 50), np.nan, data)
                            
                            valid_data = data[~np.isnan(data)]
                            if len(valid_data) > 0:
                                stats_row = {
                                    'city': city_name,
                                    'day_night': lst_time,
                                    'time_period_type': 'yearly',
                                    'time_period': year,
                                    'is_climatology': False,
                                    'lst_min': float(np.min(valid_data)),
                                    'lst_max': float(np.max(valid_data)),
                                    'lst_mean': float(np.mean(valid_data)),
                                    'lst_median': float(np.median(valid_data)),
                                    'lst_std': float(np.std(valid_data)),
                                    'lst_range': float(np.max(valid_data) - np.min(valid_data)),
                                    'valid_pixel_count': len(valid_data)
                                }
                                all_city_stats.append(stats_row)
                                print(f"  ✓ {year} ({lst_time}): median={stats_row['lst_median']:.2f}°C")
                    except Exception as e:
                        print(f"  ✗ Error processing {file_path}: {e}")
                
                # Process monthly climatology files
                monthly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_MonthlyClimatology_*.tif")
                monthly_files = sorted(glob.glob(monthly_pattern))
                
                for file_path in monthly_files:
                    try:
                        month_match = re.search(r'MonthlyClimatology_(\d+)', file_path)
                        if not month_match:
                            continue
                        month = int(month_match.group(1))
                        
                        # Read GeoTIFF and compute statistics
                        with rasterio.open(file_path) as src:
                            data = src.read(1)
                            if src.nodata is not None:
                                data = np.where(data == src.nodata, np.nan, data)
                            # Apply realistic temperature bounds for Nigerian cities
                            data = np.where((data < 10) | (data > 50), np.nan, data)
                            
                            valid_data = data[~np.isnan(data)]
                            if len(valid_data) > 0:
                                stats_row = {
                                    'city': city_name,
                                    'day_night': lst_time,
                                    'time_period_type': 'monthly',
                                    'time_period': month,
                                    'is_climatology': True,
                                    'lst_min': float(np.min(valid_data)),
                                    'lst_max': float(np.max(valid_data)),
                                    'lst_mean': float(np.mean(valid_data)),
                                    'lst_median': float(np.median(valid_data)),
                                    'lst_std': float(np.std(valid_data)),
                                    'lst_range': float(np.max(valid_data) - np.min(valid_data)),
                                    'valid_pixel_count': len(valid_data)
                                }
                                all_city_stats.append(stats_row)
                    except Exception as e:
                        print(f"  ✗ Error processing {file_path}: {e}")
            
        if all_city_stats:
            # Create DataFrame from computed statistics
            all_stats_df = pd.DataFrame(all_city_stats)
            
            # Save the combined file
            all_stats_df.to_csv(target_stats_file, index=False)
            print(f"\n✅ Created statistics file with {len(all_stats_df)} records: {target_stats_file}")
            file_found = True
        else:
            print("\n⚠️ No statistics could be computed from GeoTIFF files")
            
    # Use the new target file location
    all_stats_file = target_stats_file
    
    if os.path.exists(all_stats_file) and file_found:
        all_stats_df = pd.read_csv(all_stats_file)
        
        # Prepare data for summary table with enhanced statistics
        summary_rows = []
        
        # Process each city
        for city_name, aoi_info in aoi_data.items():
            # Filter for this city's data
            city_df = all_stats_df[all_stats_df['city'] == city_name]
            
            if city_df.empty:
                print(f"⚠️ No data found for {city_name} in the statistics file")
                continue
                
            # Process both day and night
            for lst_time in lst_time_options:
                # Filter for day/night data
                time_df = city_df[city_df['day_night'] == lst_time]
                
                if time_df.empty:
                    print(f"⚠️ No {lst_time}-time data found for {city_name}")
                    continue
                
                time_str = lst_time.capitalize()
                
                # --- MONTHLY CLIMATOLOGY STATISTICS ---
                monthly_df = time_df[(time_df['time_period_type'] == 'monthly') & (time_df['is_climatology'] == True)]
                
                if monthly_df.empty:
                    monthly_df = time_df[time_df['time_period_type'] == 'monthly']
                    print(f"⚠️ No climatology data found for {city_name} {lst_time}, using regular monthly data")
                
                if not monthly_df.empty:
                    # Calculate overall statistics
                    mean_lst = monthly_df['lst_mean'].mean()
                    median_lst = monthly_df['lst_median'].mean()  # Average of the monthly medians
                    min_lst = monthly_df['lst_mean'].min()
                    max_lst = monthly_df['lst_mean'].max()
                    lst_range = max_lst - min_lst
                    overall_min = monthly_df['lst_min'].min()  # Absolute minimum LST
                    overall_max = monthly_df['lst_max'].max()  # Absolute maximum LST
                    
                    # Calculate additional statistics
                    lst_std = monthly_df['lst_mean'].std()  # Standard deviation of monthly means
                    warmest_quarter = monthly_df.nlargest(3, 'lst_mean')['lst_mean'].mean()  # Mean of 3 warmest months
                    coolest_quarter = monthly_df.nsmallest(3, 'lst_mean')['lst_mean'].mean()  # Mean of 3 coolest months
                    seasonal_amplitude = warmest_quarter - coolest_quarter
                    
                    # Find hottest month (using mean LST)
                    hottest_idx = monthly_df['lst_mean'].idxmax()
                    hottest_month_num = monthly_df.loc[hottest_idx, 'time_period']
                    hottest_month_lst = monthly_df.loc[hottest_idx, 'lst_mean']
                    
                    # Month with highest maximum temperature
                    max_temp_idx = monthly_df['lst_max'].idxmax()
                    max_temp_month_num = monthly_df.loc[max_temp_idx, 'time_period']
                    max_temp_value = monthly_df.loc[max_temp_idx, 'lst_max']
                    
                    # Convert month numbers to names
                    month_names_dict = {
                        1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                        7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
                    }
                    hottest_month = month_names_dict.get(hottest_month_num, str(hottest_month_num))
                    max_temp_month = month_names_dict.get(max_temp_month_num, str(max_temp_month_num))
                    
                    # Find coolest month (using mean LST)
                    coolest_idx = monthly_df['lst_mean'].idxmin()
                    coolest_month_num = monthly_df.loc[coolest_idx, 'time_period']
                    coolest_month_lst = monthly_df.loc[coolest_idx, 'lst_mean']
                    coolest_month = month_names_dict.get(coolest_month_num, str(coolest_month_num))
                    
                    # --- YEARLY STATISTICS ---
                    yearly_df = time_df[time_df['time_period_type'] == 'yearly']
                    
                    # Initialize yearly statistics
                    hottest_year = "N/A"
                    hottest_year_lst = 0
                    coolest_year = "N/A"
                    coolest_year_lst = 0
                    warming_rate = "N/A"
                    trend_significance = "N/A"
                    
                    if not yearly_df.empty:
                        # Find hottest year
                        hottest_year_idx = yearly_df['lst_mean'].idxmax()
                        hottest_year = yearly_df.loc[hottest_year_idx, 'time_period']
                        hottest_year_lst = yearly_df.loc[hottest_year_idx, 'lst_mean']
                        
                        # Find coolest year
                        coolest_year_idx = yearly_df['lst_mean'].idxmin()
                        coolest_year = yearly_df.loc[coolest_year_idx, 'time_period']
                        coolest_year_lst = yearly_df.loc[coolest_year_idx, 'lst_mean']
                        
                        # Calculate warming rate if we have enough years
                        if len(yearly_df) > 3:
                            # Sort by year
                            yearly_df = yearly_df.sort_values('time_period')
                            
                            # Simple linear regression to calculate warming rate
                            years = yearly_df['time_period'].values
                            temps = yearly_df['lst_mean'].values
                            
                            if len(years) == len(temps) and len(years) > 1:
                                try:
                                    from scipy import stats
                                    # Linear regression with significance test
                                    slope, intercept, r_value, p_value, std_err = stats.linregress(years, temps)
                                    
                                    # Format warming rate with R² and significance indication
                                    r_squared = r_value**2
                                    
                                    # Check significance at 0.05 threshold
                                    if p_value < 0.05:
                                        significance = "significant"
                                        sig_marker = "*"
                                    else:
                                        significance = "not significant"
                                        sig_marker = ""
                                        
                                    warming_rate = f"{slope:.3f}°C/year (R²={r_squared:.2f}{sig_marker})"
                                    trend_significance = f"p={p_value:.3f} ({significance} at p=0.05 level)"
                                    
                                    # Also try Mann-Kendall test if available
                                    try:
                                        import pymannkendall as mk
                                        result = mk.original_test(temps)
                                        if result.p < 0.05:
                                            mk_sig = "significant"
                                            mk_marker = "*"
                                        else:
                                            mk_sig = "not significant"
                                            mk_marker = ""
                                            
                                        # Add Mann-Kendall test result
                                        trend_significance += f"\nMann-Kendall: {result.trend} trend{mk_marker} (p={result.p:.3f})"
                                    except:
                                        # Mann-Kendall test not available
                                        pass
                                        
                                except:
                                    # If scipy is not available, use numpy for a simple approach
                                    years_mean = years.mean()
                                    temps_mean = temps.mean()
                                    numerator = sum((years - years_mean) * (temps - temps_mean))
                                    denominator = sum((years - years_mean) ** 2)
                                    slope = numerator / denominator if denominator != 0 else 0
                                    warming_rate = f"{slope:.3f}°C/year"
                    
                    # Create row with enhanced statistics
                    row = {
                        'City': city_name,
                        'Time': time_str,
                        'Mean Annual LST (°C)': round(mean_lst, 1),
                        'Median LST (°C)': round(median_lst, 1),
                        'Overall Min-Max (°C)': f"{round(overall_min, 1)} - {round(overall_max, 1)}",
                        'Monthly Range (°C)': f"{round(min_lst, 1)} - {round(max_lst, 1)}",
                        'Monthly Amplitude (°C)': round(lst_range, 1),
                        'Seasonal Amplitude (°C)': round(seasonal_amplitude, 1),
                        'Std Dev (°C)': round(lst_std, 1),
                        'Hottest Month': f"{hottest_month} ({round(hottest_month_lst, 1)}°C)",
                        'Coolest Month': f"{coolest_month} ({round(coolest_month_lst, 1)}°C)",
                        'Max Temp Month': f"{max_temp_month} ({round(max_temp_value, 1)}°C)",
                        'Hottest Year': f"{hottest_year} ({round(hottest_year_lst, 1)}°C)" if hottest_year != "N/A" else "N/A",
                        'Coolest Year': f"{coolest_year} ({round(coolest_year_lst, 1)}°C)" if coolest_year != "N/A" else "N/A",
                        'Warming Rate': warming_rate,
                        'Trend Significance': trend_significance
                    }
                    
                    summary_rows.append(row)
                    
                    # Create detailed text summary for this city and time
                    summary_text = f"# {city_name} {time_str}-time LST Summary Statistics\n\n"
                    
                    try:
                        years_range = f"{TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]}"
                    except:
                        years_range = "Study Period"
                    
                    summary_text += f"## Monthly Climatology ({years_range})\n"
                    summary_text += f"- Mean annual LST: {mean_lst:.1f}°C\n"
                    summary_text += f"- Median annual LST: {median_lst:.1f}°C\n"
                    summary_text += f"- Overall temperature range: {overall_min:.1f}°C - {overall_max:.1f}°C\n"
                    summary_text += f"- Monthly mean range: {min_lst:.1f}°C - {max_lst:.1f}°C\n"
                    summary_text += f"- Monthly amplitude: {lst_range:.1f}°C\n"
                    summary_text += f"- Seasonal amplitude: {seasonal_amplitude:.1f}°C\n"
                    summary_text += f"- Standard deviation: {lst_std:.1f}°C\n"
                    summary_text += f"- Hottest month: {hottest_month} ({hottest_month_lst:.1f}°C)\n"
                    summary_text += f"- Coolest month: {coolest_month} ({coolest_month_lst:.1f}°C)\n"
                    summary_text += f"- Month with highest max temperature: {max_temp_month} ({max_temp_value:.1f}°C)\n"
                    
                    if hottest_year != "N/A":
                        summary_text += f"\n## Yearly Statistics\n"
                        summary_text += f"- Hottest year: {hottest_year} ({hottest_year_lst:.1f}°C)\n"
                        summary_text += f"- Coolest year: {coolest_year} ({coolest_year_lst:.1f}°C)\n"
                        summary_text += f"- Warming trend: {warming_rate}\n"
                        summary_text += f"- Trend significance: {trend_significance}\n"
                        summary_text += f"  * p-value < 0.05 indicates a statistically significant trend at 95% confidence level\n"
                    
                    # Save city-specific summary
                    city_dir = aoi_info['city_dir']
                    
                    # Use new directory structure with "Stats" as the base folder
                    city_stats_dir = os.path.join(main_stats_dir, city_name)
                    os.makedirs(city_stats_dir, exist_ok=True)
                    
                    # Create subfolders for day and night statistics
                    city_time_stats_dir = os.path.join(city_stats_dir, time_str)
                    os.makedirs(city_time_stats_dir, exist_ok=True)
                    
                    # Save enhanced summary text file - FIX: Add encoding parameter
                    summary_file = os.path.join(city_time_stats_dir, f"{city_name}_{time_str}_Enhanced_Summary.txt")
                    with open(summary_file, 'w', encoding='utf-8') as f:
                        f.write(summary_text)
                    print(f"  ✅ Saved enhanced {time_str} summary for {city_name} to {summary_file}")
        
        # Create enhanced summary dataframe
        if summary_rows:
            summary_df = pd.DataFrame(summary_rows)
            
            # Save enhanced CSV
            summary_csv_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.csv")
            summary_df.to_csv(summary_csv_path, index=False)
            print(f"✅ Saved enhanced summary to {summary_csv_path}")
            
            # Display the enhanced summary table
            print("\n📍 ENHANCED CITY LST SUMMARY STATISTICS")
            for city_name, aoi_info in aoi_data.items():
                city_rows = [row for row in summary_rows if row['City'] == city_name]
                
                if city_rows:
                    print(f"\n📍 {city_name}:")
                    for row in city_rows:
                        time_str = row['Time']
                        print(f"  • {time_str}-time:")
                        print(f"    • Mean annual LST: {row['Mean Annual LST (°C)']}°C")
                        print(f"    • Monthly range: {row['Monthly Range (°C)']}")
                        print(f"    • Hottest month: {row['Hottest Month']}")
                        print(f"    • Coolest month: {row['Coolest Month']}")
                        
                        if row['Hottest Year'] != "N/A":
                            print(f"    • Hottest year: {row['Hottest Year']}")
                            print(f"    • Coolest year: {row['Coolest Year']}")
                            print(f"    • Warming rate: {row['Warming Rate']}")
                            print(f"    • Significance: {row['Trend Significance']}")
            
            # Also create a styled HTML output for better visualization
            styled_summary = summary_df.style.background_gradient(
                subset=['Mean Annual LST (°C)', 'Monthly Amplitude (°C)', 'Seasonal Amplitude (°C)'],
                cmap='RdYlBu_r'
            )
            
            # Create a function to add significance markers to cells
            def highlight_significant_trends(val):
                if isinstance(val, str) and '*' in val:
                    return 'color: #008800; font-weight: bold'  # Green for significant
                return ''
            
            # Apply styling for significant trends
            styled_summary = styled_summary.applymap(
                highlight_significant_trends, 
                subset=['Warming Rate']
            )
            
            # Save HTML version
            html_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.html")
            styled_summary.to_html(html_path)
            print(f"✅ Saved HTML summary with styling to {html_path}")
    else:
        print(f"❌ Could not find or create statistics file: {all_stats_file}")
        print(f"Make sure you have run Block 3 to generate the statistics files first.")
        
except Exception as e:
    print(f"❌ Error creating enhanced summary with yearly statistics: {e}")
    traceback_str = traceback.format_exc()
    print(f"📋 Traceback: {traceback_str}")

# Calculate execution time
final_stats_end_time = time.time()
final_stats_duration = final_stats_end_time - final_stats_start_time
print("\n" + "="*70)
print(f"🎯 FINAL STATISTICS ENHANCEMENTS COMPLETED IN {final_stats_duration:.2f} SECONDS ({final_stats_duration/60:.2f} MINUTES)")
print("="*70)
print("Enhanced LST summary statistics with yearly data have been generated")
print("="*70)

## Block 7: Trend Analysis & Visualization

Creates dual-panel diagnostic plots for each city:

**Left Panel: Annual Trend**
- Time series of yearly mean LST (2006-2025)
- Linear regression trend line: $y = \beta_0 + \beta_1 \cdot t$
- 95% confidence interval (shaded region)
- Warming rate: $\Delta T = \beta_1 \times 10$ °C/decade
- Statistical significance: $p$-value and $R^2$
- Mann-Kendall test: $Z$-statistic and trend direction

**Right Panel: Monthly Climatology**
- Seasonal cycle (12 months)
- Shaded envelope: Min-Max temperature range
- Annotated peaks: Hottest and coolest months
- Summary statistics in text box

**Color Scheme:**
- Day LST: Orange (#FF7F0E)
- Night LST: Black (#000000)
- Confidence bands: Semi-transparent fill

**Significance Markers:**
- (*) indicates $p < 0.05$
- Mann-Kendall trend: ↑ (increasing), ↓ (decreasing), → (no trend)

In [ ]:
print("="*70)
print("BLOCK 7: LST TREND ANALYSIS - IMPROVED VISUALIZATION")
print("="*70)

# Define the figure size for 1x2 plots (side by side layout with shared y-axis)
FIGURE_SIZE = (18, 9)  # Larger figure for better readability
DPI = 300

# Define colors using proper matplotlib RGBA format
DAY_COLOR = '#FF7F0E'  # Orange for day
DAY_FILL = (1.0, 0.5, 0.055, 0.3)  # Lighter orange with alpha=0.3
NIGHT_COLOR = '#000000'  # Black for night line and markers
NIGHT_FILL = (0.5, 0.5, 0.5, 0.3)  # Grey with alpha=0.3 for night shaded area

# Month abbreviations
MONTH_ABBRS = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

def find_safe_text_position(ax, x, y, default_offset, text_height_fraction=0.1):
    """Find a safe position for text annotation within plot bounds"""
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Calculate text height based on y-axis range
    y_range = ylim[1] - ylim[0]
    text_height = y_range * text_height_fraction
    
    # Try default position first
    new_y = y + default_offset
    
    # If text would go outside plot area, flip it
    if new_y + text_height > ylim[1]:
        new_y = y - abs(default_offset) - text_height
    elif new_y - text_height < ylim[0]:
        new_y = y + abs(default_offset) + text_height
        
    # Ensure we stay within bounds with some padding
    new_y = min(new_y, ylim[1] - text_height*0.5)
    new_y = max(new_y, ylim[0] + text_height*0.5)
    
    return new_y

def place_trend_box_safely(ax, day_data=None, night_data=None):
    """Safely place trend boxes for both day and night data
    
    Day data always at top-left, Night data always at bottom-left
    """
    positions = {}
    
    if day_data is not None:
        positions['day'] = (0.02, 0.95)  # Top-left for day
        
    if night_data is not None:
        positions['night'] = (0.01, 0.03)  # Always bottom-left for night
    
    return positions

def place_monthly_stats_box_safely(ax, day_data=None, night_data=None):
    """Safely place monthly statistics boxes in the right panel
    
    Day data always at top-right, Night data always at bottom-right
    """
    positions = {}
    
    if day_data is not None:
        positions['day'] = (0.98, 0.95)  # Top-right for day
        
    if night_data is not None:
        positions['night'] = (0.99, 0.03)  # Always bottom-right for night
    
    return positions

try:
    # Set non-interactive matplotlib backend to avoid parallel execution errors
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.ticker as ticker  # For integer years formatting
    
    print("\n📊 Generating improved LST trend line plots for each city...")
    
    # Load the statistics file from Stats folder
    stats_file = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'All_Cities_LST_Statistics.csv')
    
    if os.path.exists(stats_file):
        all_stats_df = pd.read_csv(stats_file)
        print(f"✅ Loaded statistics from: {stats_file}")
    else:
        # Try alternative locations
        alt_stats_files = [
            os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'City_LST_Enhanced_Summary.csv'),
            os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', 'All_Cities_LST_Statistics.csv')
        ]
        
        stats_found = False
        for file in alt_stats_files:
            if os.path.exists(file):
                all_stats_df = pd.read_csv(file)
                print(f"✅ Loaded statistics from alternative location: {file}")
                stats_found = True
                break
                
        if not stats_found:
            raise FileNotFoundError(f"Could not find statistics file in any known location.")
    
    # Determine which LST time periods to process
    lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]
    
    # Process each city
    for city_name, aoi_info in aoi_data.items():
        print(f"\n📊 Processing trend plots for {city_name}...")
        
        # Filter for this city's data
        city_df = all_stats_df[all_stats_df['city'] == city_name]
        
        if city_df.empty:
            print(f"⚠️ No data found for {city_name} in the statistics file")
            continue
            
        # Create directory for trend plots in the city's LST_Temporal_Analysis folder
        city_dir = aoi_info['city_dir']
        city_temporal_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f'City_{city_name}', 'LST_Temporal_Analysis')
        os.makedirs(city_temporal_dir, exist_ok=True)
        
        # Create the figure with 1x2 subplots sharing y-axis
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGURE_SIZE, sharey=True)
        
        # Reduce space between subplots
        plt.subplots_adjust(wspace=0.05)
        
        # Get study period for title
        try:
            years_range = f"({TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]})"
        except:
            years_range = ""
            
        # Add overall title with increased font size
        fig.suptitle(f"{city_name} Land Surface Temperature {years_range}", 
                    fontweight='bold', fontsize=27, y=0.95)  # Increased from 18 to 27
        
        # Store data for both plots to determine y-axis limits and safe positioning
        all_temps = []
        day_data = {}
        night_data = {}
        
        # LEFT PLOT: Yearly Trends (both day and night)
        ax1.set_title("Yearly LST Trends", fontsize=21, pad=20)  # Increased from 14 to 21
        
        # Process yearly data for both day and night
        for lst_time in lst_time_options:
            # Filter for day/night data
            time_df = city_df[city_df['day_night'] == lst_time]
            
            if time_df.empty:
                print(f"⚠️ No {lst_time}-time data found for {city_name}")
                continue
                
            # Get yearly data
            yearly_df = time_df[time_df['time_period_type'] == 'yearly']
            
            if yearly_df.empty:
                print(f"⚠️ No yearly data found for {city_name} {lst_time}")
                continue
                
            # Sort yearly data chronologically
            yearly_df = yearly_df.sort_values('time_period')
            
            # Convert years to integers and get median LST values
            years = yearly_df['time_period'].values.astype(int)
            temps = yearly_df['lst_median'].values
            
            # Get min-max values for shaded area
            min_temps = yearly_df['lst_min'].values
            max_temps = yearly_df['lst_max'].values
            
            # Store all temperatures for y-axis limits
            all_temps.extend(min_temps)
            all_temps.extend(max_temps)
            
            # Get color and fill based on day/night
            if lst_time == 'day':
                color = DAY_COLOR
                fill_color = DAY_FILL
                linestyle = ':'  # Dotted line for day trend
                day_data = {
                    'years': years, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps
                }
            else:
                color = NIGHT_COLOR
                fill_color = NIGHT_FILL
                linestyle = ':'  # Dotted line for night trend
                night_data = {
                    'years': years, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps
                }
            
            time_label = lst_time.capitalize()
            
            # Plot shaded min-max range
            ax1.fill_between(years, min_temps, max_temps, color=fill_color, zorder=1)
            
            # Plot median line and points
            ax1.plot(years, temps, color=color, alpha=0.8, linestyle='-', linewidth=3.75, zorder=2)  # Increased from 2.5 to 3.75
            ax1.scatter(years, temps, color=color, s=60, zorder=3)  # Increased from 40 to 60
            
            # Calculate and plot trendline
            try:
                from scipy import stats
                slope, intercept, r_value, p_value, std_err = stats.linregress(years, temps)
                r_squared = r_value**2
                
                # Generate trend line
                trend_line = intercept + slope * years
                
                # Plot trendline with appropriate style
                ax1.plot(years, trend_line, color=color, linestyle=linestyle, linewidth=3, zorder=1)  # Increased from 2 to 3
                
                # Calculate total warming/cooling over the period
                total_change = slope * (years[-1] - years[0])
                warming_per_decade = slope * 10
                
                # Create trend text box with increased font size
                equation_text = f"{time_label}: y = {slope:.4f}x + {intercept:.2f}"
                stats_text = f"R² = {r_squared:.3f}, p<{p_value:.2f}*" if p_value < 0.05 else f"R² = {r_squared:.3f}, p={p_value:.3f}"
                warming_text = f"Warming: {warming_per_decade:.2f}°C/decade ({total_change:.2f}°C total)"
                
                full_text = f"{equation_text}\n{stats_text}\n{warming_text}"
                
                # Store trend info for safe positioning later
                if lst_time == 'day':
                    day_data['trend_text'] = full_text
                    day_data['color'] = color
                else:
                    night_data['trend_text'] = full_text
                    night_data['color'] = color
                
            except Exception as e:
                print(f"⚠️ Error calculating trend for {city_name} {lst_time}: {e}")
        
        # Place trend boxes safely with night always at bottom left
        box_positions = place_trend_box_safely(ax1, day_data if day_data else None, night_data if night_data else None)
        
        # Add the text boxes with fixed positions
        for lst_time in ['day', 'night']:
            data = day_data if lst_time == 'day' else night_data
            if data and 'trend_text' in data and lst_time in box_positions:
                x_pos, y_pos = box_positions[lst_time]
                
                # Set vertical alignment based on position (top for day, bottom for night)
                v_align = 'top' if lst_time == 'day' else 'bottom'
                
                ax1.text(x_pos, y_pos, data['trend_text'], transform=ax1.transAxes,
                        fontsize=15, verticalalignment=v_align, 
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.6, edgecolor=data['color'], linewidth=2),
                        color=data['color'], fontweight='bold')
        
        # Add annotations for min/max values with safe positioning
        for lst_time in ['day', 'night']:
            data = day_data if lst_time == 'day' else night_data
            if not data:
                continue
                
            years = data['years']
            temps = data['temps']
            color = DAY_COLOR if lst_time == 'day' else NIGHT_COLOR
            
            # Find latest year and minimum year values
            latest_year = years[-1]
            latest_temp = temps[-1]
            min_year_idx = np.argmin(temps)
            min_year = years[min_year_idx]
            min_temp = temps[min_year_idx]
            
            # Calculate safe positions for annotations
            y_range = ax1.get_ylim()[1] - ax1.get_ylim()[0]
            offset = y_range * 0.03  # 3% of y-range
            
            # Annotate latest year value with safe positioning
            safe_y_latest = find_safe_text_position(ax1, latest_year, latest_temp, offset)
            ax1.annotate(f"{latest_year}: {latest_temp:.1f}°C", 
                       xy=(latest_year, latest_temp),
                       xytext=(latest_year, safe_y_latest), 
                       ha='center', va='top',
                       fontsize=12, fontweight='bold', color=color,  # Increased from 10 to 15
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                       arrowprops=dict(arrowstyle='->', color=color, lw=1))
            
            # Annotate minimum year value with safe positioning
            safe_y_min = find_safe_text_position(ax1, min_year, min_temp, -offset)
            ax1.annotate(f"{min_year}: {min_temp:.1f}°C", 
                       xy=(min_year, min_temp),
                       xytext=(min_year, safe_y_min), 
                       ha='center', va='top',
                       fontsize=12, fontweight='bold', color=color,  # Increased from 10 to 15
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                       arrowprops=dict(arrowstyle='->', color=color, lw=1))
        
        # Format left plot - Ensure years are displayed as integers
        ax1.set_xlabel("Year", fontsize=18)  # Increased from 12 to 18
        ax1.set_ylabel("LST (°C)", fontsize=18)  # Increased from 12 to 18, single label
        ax1.grid(True, linestyle='--', alpha=0.6)
        ax1.tick_params(axis='both', which='major', labelsize=15)  # Increased from default to 15
        
        # Set integer years on x-axis
        ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))  # Format as integers
        
        # RIGHT PLOT: Monthly Climatology (both day and night)
        ax2.set_title("Monthly LST Climatology", fontsize=21, pad=20)  # Increased from 14 to 21
        
        # Reset data storage for monthly plot
        day_data = {}
        night_data = {}
        
        # Process monthly climatology data for both day and night
        for lst_time in lst_time_options:
            # Filter for day/night data
            time_df = city_df[city_df['day_night'] == lst_time]
            
            if time_df.empty:
                continue
                
            # Get monthly climatology data
            monthly_df = time_df[(time_df['time_period_type'] == 'monthly') & (time_df['is_climatology'] == True)]
            
            if monthly_df.empty:
                monthly_df = time_df[time_df['time_period_type'] == 'monthly']
                print(f"⚠️ No climatology data found for {city_name} {lst_time}, using regular monthly data")
                
            if monthly_df.empty:
                continue
                
            # Sort by month number
            monthly_df = monthly_df.sort_values('time_period')
            months = monthly_df['time_period'].values.astype(int)
            temps = monthly_df['lst_median'].values
            
            # Get min-max values for shaded area
            min_temps = monthly_df['lst_min'].values
            max_temps = monthly_df['lst_max'].values
            
            # Store all temperatures for consistent y-axis
            all_temps.extend(min_temps)
            all_temps.extend(max_temps)
            
            # Get color based on day/night
            if lst_time == 'day':
                color = DAY_COLOR
                fill_color = DAY_FILL
                day_data = {'months': months, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps}
            else:
                color = NIGHT_COLOR
                fill_color = NIGHT_FILL
                night_data = {'months': months, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps}
            
            # Plot shaded min-max range
            ax2.fill_between(months, min_temps, max_temps, color=fill_color, zorder=1)
            
            # Plot median line and points
            ax2.plot(months, temps, color=color, alpha=0.8, linestyle='-', linewidth=3.75, zorder=2)  # Increased from 2.5 to 3.75
            ax2.scatter(months, temps, color=color, s=60, zorder=3)  # Increased from 40 to 60
            
            # Find and annotate extreme months with safe positioning
            min_idx = np.argmin(temps)
            max_idx = np.argmax(temps)
            
            min_month = MONTH_ABBRS[months[min_idx]]
            max_month = MONTH_ABBRS[months[max_idx]]
            
            # Calculate safe positions
            y_range = np.max(all_temps) - np.min(all_temps) if all_temps else 10
            offset = y_range * 0.05  # 5% of y-range
            
            # Annotate maximum month with safe positioning
            safe_y_max = find_safe_text_position(ax2, months[max_idx], temps[max_idx], offset)
            ax2.annotate(f"{max_month}: {temps[max_idx]:.1f}°C", 
                       xy=(months[max_idx], temps[max_idx]),
                       xytext=(months[max_idx], safe_y_max),
                       ha='center', va='top',
                       fontsize=15, fontweight='bold', color=color,  # Increased from 10 to 15
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                       arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
            
            # Annotate minimum month with safe positioning
            safe_y_min = find_safe_text_position(ax2, months[min_idx], temps[min_idx], -offset)
            ax2.annotate(f"{min_month}: {temps[min_idx]:.1f}°C", 
                       xy=(months[min_idx], temps[min_idx]),
                       xytext=(months[min_idx], safe_y_min), 
                       ha='center', va='top',
                       fontsize=15, fontweight='bold', color=color,  # Increased from 10 to 15
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                       arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
            
            # Calculate additional monthly statistics
            time_label = lst_time.capitalize()
            
            # Monthly amplitude (min-max range)
            monthly_amplitude = np.max(temps) - np.min(temps)
            
            # Standard deviation
            monthly_std = np.std(temps)
            
            # Create simplified statistics text box with only MA and Std Dev
            monthly_stats_text = f"Monthly amplitude: {monthly_amplitude:.1f}°C\nStd Dev: {monthly_std:.1f}°C"
            
            # Store monthly stats info for positioning later
            if lst_time == 'day':
                day_data['monthly_stats_text'] = monthly_stats_text
                day_data['color'] = color
            else:
                night_data['monthly_stats_text'] = monthly_stats_text
                night_data['color'] = color
        
        # Place monthly statistics boxes safely with night always at bottom right
        monthly_box_positions = place_monthly_stats_box_safely(ax2, day_data if day_data else None, night_data if night_data else None)
        
        # Add monthly statistics boxes to the plot with fixed positions
        for lst_time in ['day', 'night']:
            data = day_data if lst_time == 'day' else night_data
            if data and 'monthly_stats_text' in data and lst_time in monthly_box_positions:
                x_pos, y_pos = monthly_box_positions[lst_time]
                
                # Set vertical alignment based on position (top for day, bottom for night)
                v_align = 'top' if lst_time == 'day' else 'bottom'
                
                ax2.text(x_pos, y_pos, data['monthly_stats_text'], transform=ax2.transAxes,
                        fontsize=12, verticalalignment=v_align, horizontalalignment='right',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.6, edgecolor=data['color'], linewidth=1),
                        color=data['color'], fontweight='bold')
        
        # Format right plot (no y-label since it's shared)
        ax2.set_xlabel("Month", fontsize=18)  # Increased from 12 to 18
        ax2.set_xticks(range(1, 13))
        ax2.set_xticklabels([MONTH_ABBRS[i] for i in range(1, 13)])
        ax2.grid(True, linestyle='--', alpha=0.6)
        ax2.tick_params(axis='both', which='major', labelsize=15)  # Increased from default to 15
        
        # Set consistent y-axis limits for both plots
        if all_temps:
            y_min, y_max = min(all_temps), max(all_temps)
            y_margin = (y_max - y_min) * 0.1  # Increased from 5% to 10% margin for more room
            ax1.set_ylim(y_min - y_margin, y_max + y_margin)
            ax2.set_ylim(y_min - y_margin, y_max + y_margin)
        
        # Adjust layout
        plt.tight_layout(rect=[0, 0.03, 1, 0.92])
        
        # Create directory for saving the output
        figures_dir = os.path.join(city_temporal_dir, 'Figures')
        os.makedirs(figures_dir, exist_ok=True)
        
        # Save the combined plot
        output_file_png = os.path.join(figures_dir, f"{city_name}_LST_Trends_Combined2.png")
        output_file_pdf = os.path.join(figures_dir, f"{city_name}_LST_Trends_Combined2.pdf")
        
        # Save in both formats
        plt.savefig(output_file_png, dpi=DPI, bbox_inches='tight')
        plt.savefig(output_file_pdf, bbox_inches='tight')
        
        print(f"✅ Saved trend plots for {city_name} to {figures_dir}")
        
        plt.close(fig)  # Close to free memory
    
    print("\n" + "="*70)
    print("🎯 LST TREND PLOTS COMPLETED")
    print("="*70)
    print("Trend line plots have been generated for all cities")
    print("="*70)
    
except Exception as e:
    print(f"❌ Error creating trend line plots: {e}")
    traceback_str = traceback.format_exc()
    print(f"📋 Traceback: {traceback_str}")
    
print("="*70)
print("LEGEND GENERATION FOR LST TREND PLOTS")
print("="*70)

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

try:
    # Create a standalone legend figure
    legend_fig, legend_ax = plt.subplots(figsize=(12, 3), dpi=300)
    
    # Hide the axes
    legend_ax.axis('off')
    
    # Define the legend elements matching your plot styling
    legend_elements = [
        # Day-time elements (orange)
        Line2D([0], [0], color=DAY_COLOR, linewidth=3.75, label='Day-time Median'),
        Line2D([0], [0], color=DAY_COLOR, linewidth=3, linestyle=':', label='Day Trend Line'),
        mpatches.Patch(color=DAY_FILL, label='Day Min-Max Range'),
        
        # Night-time elements (black/grey)
        Line2D([0], [0], color=NIGHT_COLOR, linewidth=3.75, label='Night-time Median'),
        Line2D([0], [0], color=NIGHT_COLOR, linewidth=3, linestyle=':', label='Night Trend Line'),
        mpatches.Patch(color=NIGHT_FILL, label='Night Min-Max Range'),
    ]
    
    # Create the legend with horizontal layout
    legend = legend_ax.legend(
        handles=legend_elements,
        loc='center',
        ncol=3,  # 6 columns to match your sample layout
        fontsize=14,
        frameon=True,
        fancybox=True,
        shadow=True,
        framealpha=0.9,
        edgecolor='black',
        facecolor='white'
    )
    
    # Adjust legend spacing
    legend.get_frame().set_linewidth(1.5)
    
    # Adjust the layout to center the legend
    legend_fig.tight_layout()
    
    # Create directory for legend output in the main stats directory
    legend_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats')
    os.makedirs(legend_dir, exist_ok=True)
    
    # Save the legend
    legend_output_png = os.path.join(legend_dir, "LST_Trend_Plot_Legend.png")
    legend_output_pdf = os.path.join(legend_dir, "LST_Trend_Plot_Legend.pdf")
    
    legend_fig.savefig(legend_output_png, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
    legend_fig.savefig(legend_output_pdf, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
    
    plt.close(legend_fig)
    
    print(f" Legend saved successfully:")
    print(f"   PNG: {legend_output_png}")
    print(f"   PDF: {legend_output_pdf}")
    
    # Also create a version for each city's figures directory
    for city_name, aoi_info in aoi_data.items():
        city_figures_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f'City_{city_name}', 
                                       'LST_Temporal_Analysis', 'Figures')
        if os.path.exists(city_figures_dir):
            city_legend_png = os.path.join(city_figures_dir, f"{city_name}_LST_Trend_Legend.png")
            city_legend_pdf = os.path.join(city_figures_dir, f"{city_name}_LST_Trend_Legend.pdf")
            
            # Copy the legend to each city's directory
            import shutil
            shutil.copy2(legend_output_png, city_legend_png)
            shutil.copy2(legend_output_pdf, city_legend_pdf)
            
            print(f"   ✓ Legend copied to {city_name} figures directory")
    
    print("\n" + "="*70)
    print("🎯 LEGEND GENERATION COMPLETED")
    print("="*70)
    print("="*70)
    
except Exception as e:
    print(f"❌ Error creating legend: {e}")
    traceback_str = traceback.format_exc()
    print(f"📋 Traceback: {traceback_str}")

## Block 8: Cross-City Heatmap Comparison

Comparative heatmaps reveal spatiotemporal patterns:

**Yearly Heatmap (Cities × Years):**
- Rows: Cities (sorted alphabetically)
- Columns: Years (2006-2025)
- Identifies warming/cooling trends across urban areas
- Highlights anomalous years (e.g., El Niño events)

**Monthly Heatmap (Cities × Months):**
- Rows: Cities
- Columns: Months (Jan-Dec)
- Reveals latitudinal climate gradients
- Compares peak temperature timing across cities

**Color Scale:**
- Diverging palette (RdYlBu_r) centered on median
- Consistent scaling for fair comparison
- Separate heatmaps for day and night LST

**Output:** Saved to `Cross_City_Statistics/` folder

In [ ]:
# =============================================================================
# HEATMAP VISUALIZATION: LST ACROSS CITIES AND TIME
# =============================================================================

print("="*70)
print("CREATING LST HEATMAPS ACROSS CITIES")
print("="*70)

# Determine which LST times to process
lst_times_to_process = []
if config.LST_CONFIG['lst_time'] in ['day', 'both']:
    lst_times_to_process.append('day')
if config.LST_CONFIG['lst_time'] in ['night', 'both']:
    lst_times_to_process.append('night')

# Load the combined statistics file
combined_stats_file = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'All_Cities_LST_Statistics.csv'
)

if not os.path.exists(combined_stats_file):
    raise FileNotFoundError(f"Combined statistics file not found: {combined_stats_file}")

all_stats_df = pd.read_csv(combined_stats_file)

# Prepare data for heatmaps
yearly_heatmaps = {}
monthly_heatmaps = {}

for lst_time in lst_times_to_process:
    # Filter data for this LST time
    lst_data = all_stats_df[all_stats_df['day_night'] == lst_time].copy()
    
    # Collect yearly and monthly data by city
    city_yearly_data = {}
    city_monthly_data = {}
    
    for city_name in aoi_data.keys():
        # Filter for this city
        city_data = lst_data[lst_data['city'] == city_name].copy()
        
        if not city_data.empty:
            # Yearly data: non-climatology yearly records
            yearly_df = city_data[
                (city_data['time_period_type'] == 'yearly') & 
                (city_data['is_climatology'] == False)
            ].copy()
            
            # Monthly data: USE CLIMATOLOGY (monthly averages across all years)
            monthly_df = city_data[
                (city_data['time_period_type'] == 'monthly') & 
                (city_data['is_climatology'] == True)  # Changed to True for climatology
            ].copy()
            
            # Rename and convert columns to match expected format for create_heatmap_data
            if not yearly_df.empty:
                yearly_df = yearly_df.rename(columns={'time_period': 'year', 'lst_median': 'median'})
                yearly_df['year'] = yearly_df['year'].astype(int)  # Ensure year is integer
                city_yearly_data[city_name] = yearly_df
            
            if not monthly_df.empty:
                monthly_df = monthly_df.rename(columns={'time_period': 'month', 'lst_median': 'median'})
                monthly_df['month'] = monthly_df['month'].astype(int)  # Ensure month is integer
                city_monthly_data[city_name] = monthly_df
        else:
            print(f"⚠️ No statistics data found for {city_name} ({lst_time})")
    
    # Create heatmap data
    if city_yearly_data:
        yearly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
            city_yearly_data, 
            time_dim='year', 
            stat='median'
        )
    else:
        yearly_heatmaps[lst_time] = pd.DataFrame()
    
    if city_monthly_data:
        monthly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
            city_monthly_data,
            time_dim='month',
            stat='median'
        )
    else:
        monthly_heatmaps[lst_time] = pd.DataFrame()
    
    print(f"✅ Prepared {lst_time}-time heatmap data")

# Create output directory for cross-city statistics
cross_city_stats_dir = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats'
)
# Create and save SEPARATE heatmaps for each LST time and period
for lst_time in lst_times_to_process:
    # Yearly heatmap
    if not yearly_heatmaps[lst_time].empty:
        yearly_output = os.path.join(
            cross_city_stats_dir,
            f'LST_Yearly_{lst_time.title()}_Heatmap_AllCities.png'
        )
        visualization.create_lst_heatmap(
            yearly_heatmaps[lst_time],
            lst_time=lst_time,
            time_dim='year',
            output_path=yearly_output,
            dpi=300
        )
    else:
        print(f"⚠️ Skipping yearly {lst_time} heatmap - no data available")
    
    # Monthly climatology heatmap
    if not monthly_heatmaps[lst_time].empty:
        monthly_output = os.path.join(
            cross_city_stats_dir,
            f'LST_Monthly_{lst_time.title()}_Heatmap_AllCities.png'
        )
        visualization.create_lst_heatmap(
            monthly_heatmaps[lst_time],
            lst_time=lst_time,
            time_dim='month',
            output_path=monthly_output,
            dpi=300
        )
    else:
        print(f"⚠️ Skipping monthly {lst_time} heatmap - no data available")

print("\n" + "="*70)
print("✅ LST HEATMAP ANALYSIS COMPLETED")
print(f"📊 Heatmaps saved to: {cross_city_stats_dir}")
print("="*70)

## Block 9: Stakeholder Validation Analysis

Validates measured LST trends against stakeholder perceptions:

**Survey Question:**  
*"Have you perceived temperature increases in the last 10 years?"*

**Validation Metrics:**

$$\text{Agreement (\%)} = \frac{\text{Stakeholders perceiving warming}}{\text{Total stakeholders}} \times 100$$

**Classification:**
- **Agree**: Perception matches measured trend direction
- **Disagree**: Perception contradicts measured trend
- **Analysis**: Separate validation for day and night LST

**Output Products:**
- Validated heatmaps with perception overlay
- Per-city agreement tables
- Summary statistics on perception accuracy

**Interpretation:**  
High agreement indicates stakeholders accurately perceive thermal changes. Low agreement may reflect:
- Adaptation to gradual warming
- Focus on extreme events vs. mean trends
- Difference between perceived and measured temperature

In [ ]:
# ============================================================================
# STAKEHOLDER VALIDATION: COMPARE PERCEIVED VS. ACTUAL LST TRENDS (YEARLY)
# ============================================================================
# Goal: Compare stakeholder perceptions of warming with actual LST trends
# Question asked: "Do you perceive temperature has increased in the last 10 years?"
# We compare this with yearly LST trends to calculate agreement

import numpy as np
import math

print("\n" + "="*70)
print("ANALYZING STAKEHOLDER VALIDATION (YEARLY TRENDS)")
print("="*70)

# Load survey data
survey_file = os.path.join(config.DATA_DIR, 'inputs', 'stakeholder_survey', 'master_composite.csv')
print(f"📂 Loading survey data from: {survey_file}")
survey_df = pd.read_csv(survey_file)
print(f"✅ Loaded {len(survey_df)} survey responses")

# Get list of study cities from aoi_data
study_cities = list(aoi_data.keys())
print(f"📍 Study cities: {study_cities}")

# Filter for WORKSHOP ATTENDEES ONLY (survey status = "Yes")
# This ensures we only count unique PIDs who actually attended workshops
workshop_attendees = survey_df[survey_df['survey status'] == 'Yes'].copy()

# Match cities between workshop attendees and study cities
# Use the matched_city_y column for city matching (where workshops were attended)
workshop_study = workshop_attendees[workshop_attendees['matched_city_y'].notna()].copy()
print(f"✅ Using WORKSHOP ATTENDEES ONLY for validation")
print(f"   Total workshop attendees in study cities: {workshop_study['pid'].nunique()}")

# Calculate metrics per city
# Method: Count unique WORKSHOP ATTENDEES (pid) per city
validation_results = []
total_workshop_attendees = workshop_study['pid'].nunique()

for city_name in study_cities:
    # Get WORKSHOP ATTENDEES for this city (filtered by survey status = "Yes")
    city_responses = workshop_study[workshop_study['matched_city_y'] == city_name]
    
    # Count UNIQUE workshop attendees for this city
    n_stakeholders = city_responses['pid'].nunique()
    
    if n_stakeholders > 0:
        # For warming perception, get one response per unique stakeholder
        # Use the first occurrence of each pid
        city_unique = city_responses.groupby('pid').first().reset_index()
        
        # Calculate percentage of total stakeholders
        pct_of_total = (n_stakeholders / total_workshop_attendees) * 100
        
        # Calculate percentage who perceived warming
        n_perceived_warming = (city_unique['61_temp_increase_10yrs'] == 'Yes').sum()
        pct_perceived_warming = (n_perceived_warming / n_stakeholders) * 100
        
        # Get duration of stay distribution (56_recency_with_city)
        duration_counts = city_unique['56_recency_with_city'].value_counts().to_dict()
        
        # Get LST trend significance from summary
        summary_file = os.path.join(
            config.get_output_dir('03_trend_analysis_monthly_annual'),
            'Stats',
            'City_LST_Enhanced_Summary.csv'
        )
        summary_df = pd.read_csv(summary_file)
        
        # Get Day LST trend for this city
        city_summary_day = summary_df[
            (summary_df['City'] == city_name) & 
            (summary_df['Time'] == 'Day')
        ]
        
        # Get Night LST trend for this city
        city_summary_night = summary_df[
            (summary_df['City'] == city_name) & 
            (summary_df['Time'] == 'Night')
        ]
        
        # Check Day significance
        if not city_summary_day.empty:
            trend_sig_day = city_summary_day['Trend Significance'].values[0]
            is_significant_day = 'significant' in str(trend_sig_day).lower() and 'not significant' not in str(trend_sig_day).lower()
            day_sig_text = 'Significant increase' if is_significant_day else 'No significant trend'
        else:
            is_significant_day = False
            day_sig_text = 'No data'
        
        # Check Night significance
        if not city_summary_night.empty:
            trend_sig_night = city_summary_night['Trend Significance'].values[0]
            is_significant_night = 'significant' in str(trend_sig_night).lower() and 'not significant' not in str(trend_sig_night).lower()
            night_sig_text = 'Significant increase' if is_significant_night else 'No significant trend'
        else:
            is_significant_night = False
            night_sig_text = 'No data'
        
        # Calculate overall agreement based on Day trend
        if not city_summary_day.empty:
            if is_significant_day:
                overall_agreement = pct_perceived_warming
            else:
                overall_agreement = 0 if pct_perceived_warming > 0 else 100
        else:
            overall_agreement = -1
    else:
        pct_of_total = 0
        pct_perceived_warming = 0
        overall_agreement = 0
        duration_counts = {}
        day_sig_text = 'No data'
        night_sig_text = 'No data'
    
    validation_results.append({
        'city': city_name,
        'n_stakeholders': n_stakeholders,
        'pct_of_total': pct_of_total,
        'pct_perceived_warming': pct_perceived_warming,
        'overall_agreement': overall_agreement,
        'duration_counts': duration_counts,
        'day_significance': day_sig_text,
        'night_significance': night_sig_text
    })

validation_df = pd.DataFrame(validation_results)

# Recalculate pct_of_total based on actual sum of stakeholders
actual_total_stakeholders = validation_df['n_stakeholders'].sum()
validation_df['pct_of_total'] = (validation_df['n_stakeholders'] / actual_total_stakeholders) * 100

print("\n" + "="*70)
print("STAKEHOLDER VALIDATION SUMMARY:")
print("="*70)
print(validation_df[['city', 'n_stakeholders', 'pct_of_total', 'pct_perceived_warming', 'overall_agreement']].to_string(index=False))

# Create nested sunburst/donut chart visualization
print("\n" + "="*70)
print("CREATING STAKEHOLDER VALIDATION SUNBURST CHART...")
print("="*70)

# Define colors
city_color = '#4a4a4a'
duration_colors_map = {
    'Within the last 5 years': '#FFD700',
    'Between 5 - 10 years': '#1AC9F5',
    'Between 11 - 15 years': '#32CD32',
    'Over 15 years': '#8B008B'
}
agree_color = '#0000FF'
disagree_color = '#FF0000'

# Total stakeholders in study cities only
total_stakeholders = validation_df['n_stakeholders'].sum()

# Dynamic grid layout: max 5 columns, wrap to rows (scales for 1-50 cities)
n_cities = len(validation_df[validation_df['n_stakeholders'] > 0])
max_cols = min(5, n_cities)
n_donut_rows = math.ceil(n_cities / max_cols)
fig_width = 8 * max_cols
fig_height = 8 * n_donut_rows + 3  # 8" per donut row + 3" for legend
fig = plt.figure(figsize=(fig_width, fig_height))

# Create grid: donut rows + 1 legend row
height_ratios = [5] * n_donut_rows + [1]
gs = fig.add_gridspec(n_donut_rows + 1, max_cols, height_ratios=height_ratios, hspace=0.3)

# Create axes dynamically
axes = []
for i in range(n_cities):
    row = i // max_cols
    col = i % max_cols
    axes.append(fig.add_subplot(gs[row, col]))
legend_ax = fig.add_subplot(gs[n_donut_rows, :])

city_idx = 0

for idx, city_data in validation_df.iterrows():
    city_name = city_data['city']
    n_stakeholders = city_data['n_stakeholders']
    pct_of_total = city_data['pct_of_total']
    overall_agreement = city_data['overall_agreement']
    duration_counts = city_data['duration_counts']
    
    if n_stakeholders == 0:
        continue
    
    ax = axes[city_idx]
    
    # Prepare data for this city's three layers
    duration_categories = ['Within the last 5 years', 'Between 5 - 10 years', 
                          'Between 11 - 15 years', 'Over 15 years']
    
    # Layer 1 (Inner): City proportion - this city vs all others (like Monthly chart)
    layer1_sizes = [n_stakeholders, total_stakeholders - n_stakeholders]
    layer1_colors = ['#FFA500', '#E0E0E0']  # Orange for this city, grey for others
    
    # Layer 2 (Middle): Agreement/Disagreement - using percentages
    agree_pct = overall_agreement
    disagree_pct = 100 - overall_agreement if not np.isnan(overall_agreement) else 0
    layer2_sizes = [agree_pct, disagree_pct] if not np.isnan(overall_agreement) else [0, 0]
    layer2_colors = [agree_color, disagree_color]
    
    # Layer 3 (Outer): Duration breakdown
    layer3_sizes = []
    layer3_colors = []
    
    for dur_cat in duration_categories:
        count = duration_counts.get(dur_cat, 0)
        if pd.isna(count):
            count = 0
        if count > 0:
            layer3_sizes.append(float(count))
            layer3_colors.append(duration_colors_map[dur_cat])
    
    # Plot three-layer donut
    if sum(layer1_sizes) > 0:
        wedges1, _ = ax.pie(layer1_sizes, radius=0.5, colors=layer1_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    if sum(layer2_sizes) > 0:
        wedges2, _ = ax.pie(layer2_sizes, radius=0.7, colors=layer2_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    if sum(layer3_sizes) > 0:
        wedges3, _ = ax.pie(layer3_sizes, radius=0.9, colors=layer3_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    # Add center text with city name and count
    ax.text(0, 0.15, city_name, ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0, -0.05, f"{n_stakeholders}", ha='center', va='center', 
           fontsize=32, fontweight='bold')
    ax.text(0, -0.25, f"({pct_of_total:.1f}%)", ha='center', va='center', fontsize=14)
    
    # Add title above each donut showing Day and Night LST significance
    title_text = f"Day: {city_data['day_significance']}, Night: {city_data['night_significance']}"
    ax.text(0.5, 0.92, title_text, transform=ax.transAxes,
           fontsize=12, ha='center', va='top',
           bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray', linewidth=1))
    
    # Set aspect ratio to be equal
    ax.set_aspect('equal')
    
    city_idx += 1

# Create shared legend
legend_ax.axis('off')

legend_elements = []

# Agreement section
legend_elements.append(plt.Line2D([0], [0], color='w', label='Agreement:', markerfacecolor='none'))
legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=agree_color, markersize=20,
                                 label='  Agrees with LST trend'))
legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=disagree_color, markersize=20,
                                 label='  Disagrees with LST trend'))

# Duration section
legend_elements.append(plt.Line2D([0], [0], color='w', label=''))
legend_elements.append(plt.Line2D([0], [0], color='w', label='Duration of Stay:', markerfacecolor='none'))
legend_elements.append(plt.Line2D([0], [0], color='w', label=''))

legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=duration_colors_map['Within the last 5 years'], 
                                 markersize=20,
                                 label='  Within the last 5 years'))
legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=duration_colors_map['Between 5 - 10 years'], 
                                 markersize=20,
                                 label='  Between 5 - 10 years'))

legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=duration_colors_map['Between 11 - 15 years'], 
                                 markersize=20,
                                 label='  Between 11 - 15 years'))
legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                 markerfacecolor=duration_colors_map['Over 15 years'], 
                                 markersize=20,
                                 label='  Over 15 years'))

legend = legend_ax.legend(handles=legend_elements, loc='center', ncol=3,
                         frameon=False, fontsize=14, handletextpad=0.5, columnspacing=2)

fig.suptitle(
    f"Annual Validation: Multi-Layer Donuts (Total: {total_stakeholders} stakeholders)\n"
    f"Inner: City Proportion | Middle: Annual Trend Agreement | Outer: Duration",
    fontsize=16, fontweight='bold', y=0.98
)

# Save figure
validation_chart_path = os.path.join(cross_city_stats_dir, 'Stakeholder_Validation_Sunburst_Chart.png')
plt.savefig(validation_chart_path, dpi=300, bbox_inches='tight')
plt.savefig(validation_chart_path.replace('.png', '.pdf'), bbox_inches='tight')
plt.close()

print("\n" + "="*70)
print("✅ Stakeholder validation sunburst chart saved:")
print(f"   {validation_chart_path}")

print("\n" + "="*70)
print("✅ STAKEHOLDER VALIDATION COMPLETED")
print("="*70)

In [ ]:
# ============================================================================
# BLOCK 10: MONTHLY TREND STAKEHOLDER VALIDATION
# ============================================================================

print("\n" + "="*70)
print("MONTHLY TREND STAKEHOLDER VALIDATION")
print("="*70)

# Load LST summary for monthly peaks (Day/Night)
city_summary = pd.read_csv(summary_csv_path)

# Create validation results for monthly trends
monthly_validation_results = []

for city in study_cities:
    # Get city-specific responses
    city_responses = workshop_study[workshop_study['matched_city_y'] == city].copy()
    
    if len(city_responses) == 0:
        continue
    
    # Count stakeholders
    n_stakeholders = city_responses['pid'].nunique()
    
    # Get one response per unique stakeholder (to avoid counting duplicates)
    city_unique = city_responses.groupby('pid').first().reset_index()
    
    # Get LST monthly peaks from summary for BOTH Day and Night
    city_lst_day = city_summary[(city_summary['City'] == city) & (city_summary['Time'] == 'Day')]
    city_lst_night = city_summary[(city_summary['City'] == city) & (city_summary['Time'] == 'Night')]
    
    if city_lst_day.empty and city_lst_night.empty:
        continue
    
    # Map month abbreviations to full months
    month_map = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March', 'Apr': 'April',
                 'May': 'May', 'Jun': 'June', 'Jul': 'July', 'Aug': 'August',
                 'Sep': 'September', 'Oct': 'October', 'Nov': 'November', 'Dec': 'December'}
    
    # Extract hottest months from BOTH Day and Night LST analysis
    lst_day_hottest = None
    lst_night_hottest = None
    
    if not city_lst_day.empty:
        lst_day_abbr = city_lst_day['Hottest Month'].values[0].split(' ')[0]  # e.g., "Mar (40.5°C)" -> "Mar"
        lst_day_hottest = month_map.get(lst_day_abbr, lst_day_abbr)
    
    if not city_lst_night.empty:
        lst_night_abbr = city_lst_night['Hottest Month'].values[0].split(' ')[0]
        lst_night_hottest = month_map.get(lst_night_abbr, lst_night_abbr)
    
    # Define month range mappings for stakeholder responses
    # Survey uses ranges like "March – May", "December – February", etc.
    month_ranges = {
        'March – May': ['March', 'April', 'May'],
        'December – February': ['December', 'January', 'February'],
        'June – August': ['June', 'July', 'August'],
        'September – November': ['September', 'October', 'November']
    }
    
    # Calculate agreement with monthly peak identification
    # Survey column: 59_temp_peak_descr (month range when temperature peaks)
    # ENHANCED AGREEMENT LOGIC:
    #   - Stakeholders select RANGES (e.g., "March – May" covers March, April, May)
    #   - LST analysis identifies hottest months for BOTH Day and Night
    #   - Agreement = LST hottest month (Day OR Night) FALLS WITHIN stakeholder's selected range
    #   - This accounts for people experiencing heat during both day and night
    peak_month_responses = city_unique['59_temp_peak_descr'].dropna()
    
    if len(peak_month_responses) > 0:
        agree_count = 0
        disagree_count = 0
        agree_day_count = 0
        agree_night_count = 0
        agree_both_count = 0
        
        for response in peak_month_responses:
            # Check if LST hottest month falls within stakeholder's indicated range
            if response in month_ranges:
                day_agrees = lst_day_hottest in month_ranges[response] if lst_day_hottest else False
                night_agrees = lst_night_hottest in month_ranges[response] if lst_night_hottest else False
                
                if day_agrees or night_agrees:
                    agree_count += 1  # Stakeholder's range includes at least one hottest month
                    if day_agrees and night_agrees:
                        agree_both_count += 1
                    elif day_agrees:
                        agree_day_count += 1
                    else:
                        agree_night_count += 1
                else:
                    disagree_count += 1  # Stakeholder's range does NOT include either hottest month
            else:
                # If response format doesn't match expected ranges, count as uncertain (not counted)
                pass
        
        total_responses = agree_count + disagree_count
        overall_agreement = (agree_count / total_responses) * 100 if total_responses > 0 else 0
    else:
        overall_agreement = np.nan
        agree_day_count = 0
        agree_night_count = 0
        agree_both_count = 0
    
    # Duration of stay breakdown - use city_unique to avoid duplicates
    duration_col = '56_recency_with_city'
    duration_counts = city_unique[duration_col].value_counts().to_dict()
    
    monthly_validation_results.append({
        'city': city,
        'n_stakeholders': n_stakeholders,
        'lst_day_hottest': lst_day_hottest if lst_day_hottest else 'N/A',
        'lst_night_hottest': lst_night_hottest if lst_night_hottest else 'N/A',
        'overall_agreement': overall_agreement,
        'agree_day_only': agree_day_count,
        'agree_night_only': agree_night_count,
        'agree_both': agree_both_count,
        'duration_counts': duration_counts
    })

# Create DataFrame
monthly_validation_df = pd.DataFrame(monthly_validation_results)

# Calculate percentage of total
actual_total_monthly = monthly_validation_df['n_stakeholders'].sum()
monthly_validation_df['pct_of_total'] = (monthly_validation_df['n_stakeholders'] / actual_total_monthly) * 100

print("\n" + "="*70)
print("MONTHLY PEAK VALIDATION SUMMARY")
print("="*70)
print(f"Total stakeholders across study cities: {actual_total_monthly}")
print()
print("ENHANCED AGREEMENT METHODOLOGY:")
print("  • Stakeholders selected MONTH RANGES (e.g., 'March-May', 'December-February')")
print("  • LST analysis identifies hottest months for BOTH Day and Night")
print("  • Agreement = LST hottest month (Day OR Night) FALLS WITHIN stakeholder's selected range")
print("  • This accounts for heat perception during both daytime and nighttime")
print("  • Example: If stakeholder selected 'March-May' and Day hottest = 'April' → AGREES")
print("  • Example: If stakeholder selected 'March-May' and Night hottest = 'June' → DISAGREES")
print("  • Example: If stakeholder selected 'March-May' and Day='April', Night='May' → AGREES (both)")
print()

for _, row in monthly_validation_df.iterrows():
    print(f"{row['city']}:")
    print(f"  Stakeholders: {row['n_stakeholders']} ({row['pct_of_total']:.1f}%)")
    print(f"  Day LST Hottest Month: {row['lst_day_hottest']}")
    print(f"  Night LST Hottest Month: {row['lst_night_hottest']}")
    print(f"  Overall Agreement: {row['overall_agreement']:.1f}%")
    print(f"    - Agreed with Day only: {row['agree_day_only']}")
    print(f"    - Agreed with Night only: {row['agree_night_only']}")
    print(f"    - Agreed with Both: {row['agree_both']}")
    print()

# ============================================================================
# CREATE MONTHLY TREND VALIDATION VISUALIZATION
# ============================================================================

print("\n" + "="*70)
print("CREATING MONTHLY TREND STAKEHOLDER VALIDATION SUNBURST CHART...")
print("="*70)

# Define colors (same as annual validation)
city_color = '#4a4a4a'  # Dark grey for all cities (inner layer)
duration_colors_map = {
    'Within the last 5 years': '#FFD700',     # Gold (primary yellow)
    'Between 5 - 10 years': '#4169E1',       # Royal Blue (changed from orange to avoid confusion with disagreement red)
    'Between 11 - 15 years': '#32CD32',      # Lime Green (primary green)
    'Over 15 years': '#8B008B'               # Dark Magenta (primary purple)
}
agree_color = '#0000FF'  # Blue for agrees
disagree_color = '#FF0000'  # Red for disagrees

# Total stakeholders in study cities only
total_monthly_stakeholders = monthly_validation_df['n_stakeholders'].sum()

# Dynamic grid layout: max 5 columns, wrap to rows (scales for 1-50 cities)
n_cities_monthly = len(monthly_validation_df[monthly_validation_df['n_stakeholders'] > 0])
max_cols = min(5, n_cities_monthly)
n_donut_rows = math.ceil(n_cities_monthly / max_cols)
fig_width = 8 * max_cols
fig_height = 8 * n_donut_rows + 3  # 8" per donut row + 3" for legend
fig_monthly = plt.figure(figsize=(fig_width, fig_height))

# Create grid: donut rows + 1 legend row
height_ratios = [4] * n_donut_rows + [1]
gs_monthly = fig_monthly.add_gridspec(n_donut_rows + 1, max_cols, height_ratios=height_ratios, hspace=0.3)

axes_monthly = []

for city_idx, (_, city_data) in enumerate(monthly_validation_df.iterrows()):
    if city_data['n_stakeholders'] == 0:
        continue
    
    # Dynamic row/col position
    grid_row = city_idx // max_cols
    grid_col = city_idx % max_cols
    
    # Create subplot
    ax = fig_monthly.add_subplot(gs_monthly[grid_row, grid_col])
    axes_monthly.append(ax)
    
    city = city_data['city']
    n_city_stakeholders = city_data['n_stakeholders']
    pct_of_total = city_data['pct_of_total']
    overall_agreement = city_data['overall_agreement']
    
    # Layer 1: City proportion (innermost)
    layer1_sizes = [n_city_stakeholders, total_monthly_stakeholders - n_city_stakeholders]
    layer1_colors = ['#FFA500', '#E0E0E0']  # Orange for city, grey for others
    
    # Layer 2: Agreement/Disagreement (middle)
    agree_pct = overall_agreement
    disagree_pct = 100 - overall_agreement if not np.isnan(overall_agreement) else 0
    layer2_sizes = [agree_pct, disagree_pct] if not np.isnan(overall_agreement) else [0, 0]
    layer2_colors = [agree_color, disagree_color]
    
    # Layer 3: Duration breakdown (outermost)
    duration_categories = ['Within the last 5 years', 'Between 5 - 10 years', 
                          'Between 11 - 15 years', 'Over 15 years']
    layer3_sizes = []
    layer3_colors = []
    
    for dur_cat in duration_categories:
        count = city_data['duration_counts'].get(dur_cat, 0)
        if count > 0:
            layer3_sizes.append(count)
            layer3_colors.append(duration_colors_map[dur_cat])
    
    # Plot three-layer donut
    if sum(layer1_sizes) > 0:
        wedges1, _ = ax.pie(layer1_sizes, radius=0.5, colors=layer1_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    if sum(layer2_sizes) > 0:
        wedges2, _ = ax.pie(layer2_sizes, radius=0.7, colors=layer2_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    if sum(layer3_sizes) > 0:
        wedges3, _ = ax.pie(layer3_sizes, radius=0.9, colors=layer3_colors,
                           wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2),
                           startangle=90)
    
    # Add center text with city name and count
    ax.text(0, 0.15, city, ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0, -0.05, f"{n_city_stakeholders}", ha='center', va='center', 
           fontsize=32, fontweight='bold')
    ax.text(0, -0.25, f"({pct_of_total:.1f}%)", ha='center', va='center', fontsize=14)
    
    # Add title above each donut showing Day and Night hottest months
    title_text = f"Day: {city_data['lst_day_hottest']}, Night: {city_data['lst_night_hottest']}"
    ax.text(0.5, 0.92, title_text, transform=ax.transAxes,
           fontsize=12, ha='center', va='top',
           bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray', linewidth=1))

# Add main title
fig_monthly.suptitle(
    f"Monthly Peak Validation: Multi-Layer Split Donuts (Total: {total_monthly_stakeholders} stakeholders)\n"
    f"Inner: City Proportion | Middle: Monthly Peak Agreement (Day OR Night) | Outer: Duration",
    fontsize=16, fontweight='bold', y=0.98
)

# Create legend in bottom row
legend_ax_monthly = fig_monthly.add_subplot(gs_monthly[n_donut_rows, :])
legend_ax_monthly.axis('off')

legend_elements_monthly = []

# Agreement/Disagreement legend (Layer 2)
legend_elements_monthly.append(plt.Line2D([0], [0], marker='s', color='w', 
                                         markerfacecolor=agree_color, markersize=15, 
                                         label='  Agreement with Monthly Peak (Day OR Night)'))
legend_elements_monthly.append(plt.Line2D([0], [0], marker='s', color='w', 
                                         markerfacecolor=disagree_color, markersize=15, 
                                         label='  Disagreement with Monthly Peak'))

# Duration legend (Layer 3)
for dur_cat in duration_categories:
    legend_elements_monthly.append(plt.Line2D([0], [0], marker='D', color='w', 
                                              markerfacecolor=duration_colors_map[dur_cat], 
                                              markersize=15, 
                                              label=f'  {dur_cat}'))

legend_monthly = legend_ax_monthly.legend(handles=legend_elements_monthly, 
                                         loc='center', frameon=True, fontsize=14, ncol=3,
                                         title='Duration of Stay:', title_fontsize=16)

# Save figure
monthly_output = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Monthly_Sunburst_Chart'
)

plt.savefig(f"{monthly_output}.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(f"{monthly_output}.pdf", bbox_inches='tight', facecolor='white')
plt.close()

print(f"✅ Monthly validation sunburst chart saved: {monthly_output}.png")

# Save monthly validation results to CSV
monthly_summary_path = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Monthly_Summary.csv'
)

# Prepare DataFrame for export
monthly_export_df = monthly_validation_df[[
    'city', 'n_stakeholders', 'pct_of_total', 
    'lst_day_hottest', 'lst_night_hottest',
    'overall_agreement', 'agree_day_only', 'agree_night_only', 'agree_both'
]].copy()

monthly_export_df.to_csv(monthly_summary_path, index=False)
print(f"✅ Monthly validation summary saved: {monthly_summary_path}")

print("\n" + "="*70)
print("MONTHLY VALIDATION COMPLETE")
print("="*70)

---

## Summary & Next Steps

### What We Accomplished
✅ Extracted monthly and annual LST data for all study cities (2006-2025)  
✅ Applied hierarchical QA filtering for data quality assurance  
✅ Computed comprehensive temporal statistics  
✅ Performed trend analysis with statistical significance testing  
✅ Generated publication-ready visualizations  
✅ Validated trends against stakeholder perceptions  

### Key Outputs
- Monthly climatology GeoTIFFs (12 per city)
- Yearly LST GeoTIFFs (20 per city)
- Statistical summaries (CSV files)
- Trend analysis plots (PNG/PDF)
- Cross-city comparison heatmaps
- Stakeholder validation reports

### Continue To
➡️ **Notebook 04: Daily LST Analysis**
- High-resolution daily temperature patterns
- Diurnal temperature range (DTR) analysis
- Extreme heat event detection

---

**📝 Note:** All outputs from this notebook are stored in `data/outputs/03_trend_analysis_monthly_annual/`